# Transaction exceptions in the original workbook

Run from top to bottom on a CPU. Start with all original rows, including duplicates and anonymous customers. The first 17 worksheets investigate C invoices, nonpositive quantities and nonpositive prices. For zero prices, see “Which rows have a nonpositive unit price?” and “Invoice 550188”. The remaining sheets trace the signed merchandise ledger, credit matching, net retained units at two cutoffs, and observed later purchases after a return. All 27 worksheets are computed from the same original workbook.

# 原始工作簿中的交易异常

使用 CPU，从上到下运行。原始行分析包含重复明细和匿名客户。前 17 张工作表核查 C 发票、非正数量和非正价格。零价可从「哪些记录的单价非正？」和「发票 550188」开始阅读。其余各表追溯带符号商品账本、贷项分配、两个截止点的净保留数量，以及退货后能观察到的后续购买。全部 27 张表都从同一份原始工作簿重算。

## Input and output

In Colab, mount Drive and use the original `data/raw/Online Retail.xlsx`. If prompted, copy its path from the Files panel. A workbook under `data/raw/` identifies the project folder automatically. For a different layout, select the project's output folder. Locally, run from the repository root or from `docs/technical/`. Check both printed paths.

## 输入与输出

在 Colab 挂载 Drive，使用原始的 `data/raw/Online Retail.xlsx`。出现路径提示时，从左侧「文件」面板复制工作簿路径。如果它位于 `data/raw/` 下，代码会自动定位项目文件夹。其他目录结构需指定用于存放结果的项目文件夹。本地可从仓库根目录或 `docs/technical/` 运行。请核对打印的两条路径。

In [1]:
from pathlib import Path
from IPython.display import display

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is None:
    project = Path.cwd()
    if project.parts[-2:] == ("docs", "technical"):
        project = project.parents[1]
    workbook = project / "data/raw/Online Retail.xlsx"
else:
    drive.mount("/content/drive")
    project = Path("/content/drive/MyDrive/CS5481_Project2_OnlineRetail")
    workbook = project / "data/raw/Online Retail.xlsx"
    if not workbook.is_file():
        workbook = Path(input("Full workbook path / 工作簿完整路径：").strip())
        if workbook.parent.name == "raw" and workbook.parent.parent.name == "data":
            project = workbook.parents[2]
        else:
            project = Path(input("Drive project folder / Drive 项目文件夹完整路径：").strip())
            if not project.is_absolute() or not project.is_dir():
                raise ValueError("Select an existing project folder / 请选择已存在的项目文件夹")

if not workbook.is_file():
    raise FileNotFoundError(f"Workbook not found / 找不到工作簿：{workbook}")
output = project / "results/transaction_exceptions.xlsx"
print("Input / 输入：", workbook if drive else workbook.relative_to(project))
print("Output / 输出：", output if drive else output.relative_to(project))

Input / 输入： data/raw/Online Retail.xlsx
Output / 输出： results/transaction_exceptions.xlsx


## Verify the original eight fields

Check the workbook fingerprint, preserve the three identifiers as text, and keep the original Excel row number. The remaining cells run entirely inside this notebook.

## 核查原表的八个字段

先核对工作簿指纹，按文本读取三个编号列，并保留 Excel 原始行号。后面的分析代码均可在本 notebook 内直接运行。

In [2]:
import hashlib
import re
import pandas as pd

RAW_COLUMNS = [
    "InvoiceNo", "StockCode", "Description", "Quantity",
    "InvoiceDate", "UnitPrice", "CustomerID", "Country",
]
with workbook.open("rb") as source:
    digest = hashlib.sha256(source.read()).hexdigest()
    if digest != "43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d":
        raise ValueError("Check the original workbook / 请核对原始工作簿")
    source.seek(0)
    raw = pd.read_excel(source, sheet_name="Online Retail", engine="openpyxl",
                        dtype={c: "string" for c in ("InvoiceNo", "StockCode", "CustomerID")},
                        keep_default_na=False, na_values=[""])
assert raw.columns.tolist() == RAW_COLUMNS
raw.insert(0, "ExcelRow", range(2, len(raw) + 2))
tables = {}
print(f"Source rows / 原始行数：{len(raw):,}")

Source rows / 原始行数：541,909


## Count the sign combinations

From each original line derive the three flags and a signed line amount. Each row in `sign_combinations` groups lines by invoice prefix, sign of quantity, sign of unit price, and presence of a customer ID. The groups partition the worksheet. `rows` counts source rows, `invoices` counts distinct invoice numbers within that group, and `signed_gbp` sums quantity × unit price. One invoice may appear in several groups.

## 统计符号组合

由每一条原始记录计算三种标记和带符号的行金额。`sign_combinations` 按发票前缀、数量符号、单价符号和是否有客户编号分组，每条原始行恰好落入一个组合。`rows` 是原始行数，`invoices` 是组内不同发票号数，`signed_gbp` 为数量乘以单价后的带符号金额之和。同一张发票可以出现在多个组。

In [3]:
def _sign(values: pd.Series) -> pd.Series:
    signs = pd.Series("zero", index=values.index)
    signs.loc[values.gt(0)] = "positive"
    signs.loc[values.lt(0)] = "negative"
    return signs.mask(values.isna(), "missing")

lines = raw.assign(
    invoice_c=raw["InvoiceNo"].str.startswith("C", na=False),
    quantity_sign=_sign(raw["Quantity"]),
    price_sign=_sign(raw["UnitPrice"]),
    customer_present=raw["CustomerID"].notna(),
    amount_gbp=raw["Quantity"] * raw["UnitPrice"],
)
combinations = lines.groupby(
    ["invoice_c", "quantity_sign", "price_sign", "customer_present"],
    dropna=False, sort=True,
).agg(rows=("ExcelRow", "size"), invoices=("InvoiceNo", "nunique"),
      stock_codes=("StockCode", "nunique"), signed_gbp=("amount_gbp", "sum")).reset_index()
tables["sign_combinations"] = combinations
assert combinations["rows"].sum() == len(raw)
display(combinations)

,invoice_c,quantity_sign,price_sign,customer_present,rows,invoices,stock_codes,signed_gbp
0,False,negative,zero,False,1336,1336,1082,0.000
1,False,positive,negative,False,2,2,1,-22124.120
2,False,positive,positive,False,132220,1428,3408,1755276.640
3,False,positive,positive,True,397884,18532,3665,8911407.904
4,False,positive,zero,False,1139,785,654,0.000
5,False,positive,zero,True,40,34,35,0.000
6,True,negative,positive,False,383,182,161,-285470.400
7,True,negative,positive,True,8905,3654,1920,-611342.090


In [4]:
exception = lines.invoice_c | lines.Quantity.le(0) | lines.UnitPrice.le(0)
flagged = lines.loc[exception].copy()
priced = lines.loc[~lines.invoice_c & lines.Quantity.gt(0) & lines.UnitPrice.gt(0)]
invoice_counts = priced.groupby("InvoiceNo").size()
flagged["same_invoice_positive_priced_rows"] = flagged.InvoiceNo.map(invoice_counts).fillna(0).astype(int)
line_fields = ["ExcelRow", "InvoiceNo", "InvoiceDate", "StockCode", "Description",
               "Quantity", "UnitPrice", "CustomerID", "amount_gbp"]
print(f"Distinct flagged source rows / 命中任一条件的原表行数：{len(flagged):,}")

Distinct flagged source rows / 命中任一条件的原表行数：11,805


**Reading the output**

**`sign_combinations`** · The eight observed combinations of invoice prefix, quantity sign, price sign and customer-ID availability account for all 541,909 source rows exactly once. The union of C invoices, nonpositive quantities and nonpositive prices contains 11,805 distinct source rows.

**如何解读输出**

**`sign_combinations`** · 按发票前缀、数量符号、价格符号和客户编号有无分组，实际出现的八种组合恰好覆盖原表 541,909 行。C 发票、数量非正和单价非正三个条件的并集涉及 11,805 个原始行号。

## Which rows have a nonpositive unit price?

UCI defines `UnitPrice` as the product price per unit in sterling. It supplies no reason field explaining a zero price on every line. Filter the raw rows on this one field, then group by the sign of the price, the sign of the quantity, and the presence of a customer ID. There are 2,515 zero-price rows and two negative-price rows. The four groups in the output account for all 2,517 rows exactly once. `invoices` counts distinct invoice IDs within each group. [UCI Online Retail variable definitions](https://archive.ics.uci.edu/dataset/352/online+retail)

## 哪些记录的单价非正？

UCI 将 `UnitPrice` 定义为以英镑计价的商品单价。原表没有逐行解释零单价原因的字段。先按单价筛选原始记录，再按价格符号、数量符号和客户编号是否存在分组。原表中 2,515 行零价、2 行负价。输出中四组互斥，合计 2,517 行。`invoices` 统计各组内不同的发票编号数。[UCI Online Retail 字段定义](https://archive.ics.uci.edu/dataset/352/online+retail)

In [5]:
price_rows = lines.loc[lines["UnitPrice"].le(0)].copy()
price_rows["blank_description"] = price_rows["Description"].fillna("").str.strip().eq("")
price_groups = (
    price_rows.groupby(["price_sign", "quantity_sign", "customer_present"], dropna=False)
    .agg(rows=("ExcelRow", "size"), invoices=("InvoiceNo", "nunique"),
         blank_descriptions=("blank_description", "sum"))
    .reset_index()
)
assert price_groups["rows"].sum() == len(price_rows)
display(price_groups)

,price_sign,quantity_sign,customer_present,rows,invoices,blank_descriptions
0,negative,positive,False,2,2,0
1,zero,negative,False,1336,1336,862
2,zero,positive,False,1139,785,592
3,zero,positive,True,40,34,0


### What does the original wording support?

- **Zero price, negative quantity, no customer.** There are 1,336 rows, including 862 with a blank description. Literal descriptions include `damaged`, `samples`, and `Unsaleable, destroyed.`. They give different possible contexts for an inventory decrease.
- **Zero price, positive quantity, no customer.** There are 1,139 rows, including 592 with a blank description. Some entries say `found` or `adjustment`. A stock entry is plausible for those cases. A product name or blank description alone does not establish a reason.
- **Zero price, positive quantity, identified customer.** All 40 descriptions are present. Six rows have code `M` and description `Manual`. The other 34 have product descriptions. Their presence beside priced items is checked below.
- **Negative price, positive quantity, no customer.** Invoices `A563186` and `A563187` each record one unit at −£11,062.06. Both rows say `Adjust bad debt` and carry code `B`. Show both rows with the positive-price entry on `A563185` for comparison. These are accounting entries. Their signed line amounts by themselves do not establish cash settlement.

The next table counts **exact** selected descriptions, preserving case and punctuation. The examples cover only those literal strings.

### 原始文字能支持哪些解释？

- **零价、负数量、无客户编号。** 共 1,336 行，其中 862 行缺描述。原文包括 `damaged`（损坏）、`samples`（样品）、`Unsaleable, destroyed.`（无法出售，已销毁）。这些文字提示数量减少可能有不同背景。
- **零价、正数量、无客户编号。** 共 1,139 行，其中 592 行缺描述。部分原文为 `found`（找到）或 `adjustment`（调整），相应行可能记录库存操作。仅凭商品名称或空描述无法确定原因。
- **零价、正数量、有客户编号。** 40 行均有描述，其中 6 行的商品码为 `M`、描述为 `Manual`，其余 34 行是商品描述。下文核查同票中是否还有正常计价商品。
- **负价、正数量、无客户编号。** 发票 `A563186` 和 `A563187` 各记录 1 件，单价均为 −£11,062.06。两行均写有 `Adjust bad debt`（调整坏账），商品码均为 `B`。下方将两行与 `A563185` 上的正价条目并排展示。这些是账务记录，仅凭带符号的行金额无法确认实际资金结算。

下表按所选描述**精确匹配**，保留原始大小写和标点。计数只覆盖列出的文字。

In [6]:
selected_descriptions = ["damaged", "samples", "Unsaleable, destroyed.", "found", "adjustment", "Manual"]
exact_notes = price_rows.loc[price_rows["Description"].isin(selected_descriptions)]
display(
    exact_notes.groupby(["quantity_sign", "customer_present", "Description"])
    .size().rename("rows").reset_index()
)

debt_entries = lines.loc[
    lines["InvoiceNo"].isin(["A563185", "A563186", "A563187"]) & lines["StockCode"].eq("B"),
    ["ExcelRow", "InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice", "amount_gbp"],
]
display(debt_entries)

,quantity_sign,customer_present,Description,rows
0,negative,False,"Unsaleable, destroyed.",9
1,negative,False,adjustment,2
2,negative,False,damaged,42
3,negative,False,samples,1
4,positive,False,adjustment,14
5,positive,False,damaged,1
6,positive,False,found,25
7,positive,True,Manual,6


,ExcelRow,InvoiceNo,StockCode,Description,Quantity,UnitPrice,amount_gbp
299982,299984,A563185,B,Adjust bad debt,1,11062.06,11062.06
299983,299985,A563186,B,Adjust bad debt,1,-11062.06,-11062.06
299984,299986,A563187,B,Adjust bad debt,1,-11062.06,-11062.06


### Why does the final price filter account for 40 additional rows?

Recreate eligibility immediately before the price step on the **eight original fields**. A row must survive exact-duplicate removal, have a customer ID and a non-C invoice, and have positive quantity. Then apply `UnitPrice <= 0`. Comparing source row numbers verifies that the resulting 40 rows are exactly the identified, positive-quantity, zero-price group; both negative-price debt rows already failed the missing-ID rule.

For each of the 40 rows, look for another non-C, positive-quantity, positive-price line on the **same invoice**. The counts distinguish rows from invoices: the 40 rows span 34 invoices; 33 of the rows, on 30 invoices, have a priced companion. The other seven rows span four invoices that contain only zero-price lines. The detail table lists all 40 source rows. A complimentary item, promotion, or data-entry error needs additional evidence.

### 为什么最后的价格规则净排除 40 行？

使用**全部八个原始字段**复算价格步骤前的保留条件：排除完全重复行后，客户编号存在、发票号非 C、数量为正。再应用 `UnitPrice <= 0`。通过对比原始 Excel 行号，可以验证剩下的 40 行恰好是有客户编号、数量为正的零价组。两条负价坏账行已在缺失客户编号步骤排除。

对这 40 行逐行检查**同张发票**是否还包含非 C、正数量、正单价的其他记录。40 行分属 34 张发票；其中有 33 行与正常计价行同票，这 33 行分属 30 张发票。其余 7 行分属 4 张全票零价的发票。下方列出全部 40 行，可逐行核对。确认是否为赠品、促销或价格录入问题仍需其他证据。

In [7]:
duplicate = raw.duplicated(RAW_COLUMNS, keep="first")
before_price = (
    ~duplicate & raw["CustomerID"].notna()
    & ~raw["InvoiceNo"].str.startswith("C", na=False) & raw["Quantity"].gt(0)
)
final_price_exclusions = raw.loc[before_price & raw["UnitPrice"].le(0)]

price_evidence = flagged.loc[flagged["UnitPrice"].le(0)]
identified_zero = price_evidence.loc[
    price_evidence["UnitPrice"].eq(0) & price_evidence["Quantity"].gt(0)
    & price_evidence["CustomerID"].notna()
].copy()
assert set(final_price_exclusions["ExcelRow"]) == set(identified_zero["ExcelRow"])
print("Net rows removed at price step / 价格步骤净排除行数：", len(final_price_exclusions))

identified_zero["manual_entry"] = identified_zero["StockCode"].eq("M") & identified_zero["Description"].eq("Manual")
identified_zero["priced_companion"] = identified_zero["same_invoice_positive_priced_rows"].gt(0)
display(identified_zero.groupby("manual_entry").size().rename("rows").reset_index())
display(identified_zero.groupby("priced_companion").agg(
    zero_price_rows=("ExcelRow", "size"), distinct_invoices=("InvoiceNo", "nunique")
).reset_index())

display(identified_zero[[
    "ExcelRow", "InvoiceNo", "StockCode", "Description", "Quantity", "CustomerID",
    "same_invoice_positive_priced_rows", "priced_companion",
]].sort_values("ExcelRow"))

Net rows removed at price step / 价格步骤净排除行数： 40


,manual_entry,rows
0,False,34
1,True,6


,priced_companion,zero_price_rows,distinct_invoices
0,False,7,4
1,True,33,30


,ExcelRow,InvoiceNo,StockCode,Description,Quantity,CustomerID,same_invoice_positive_priced_rows,priced_companion
9302,9304,537197,22841,ROUND CAKE TIN VINTAGE GREEN,1,12647,16,True
33576,33578,539263,22580,ADVENT CALENDAR GINGHAM SACK,4,16560,20,True
40089,40091,539722,22423,REGENCY CAKESTAND 3 TIER,10,14911,28,True
47068,47070,540372,22090,PAPER BUNTING RETROSPOT,24,13081,169,True
47070,47072,540372,22553,PLASTERS IN TIN SKULLS,24,13081,169,True
56674,56676,541109,22168,ORGANISER WOOD ANTIQUE WHITE,1,15107,1,True
86789,86791,543599,84535B,FAIRY CAKES NOTEBOOK A6 SIZE,16,17560,0,False
130188,130190,547417,22062,CERAMIC BOWL WITH LOVE HEART DESIGN,36,13239,21,True
139453,139455,548318,22055,MINI CAKE STAND HANGING STRAWBERY,5,13113,6,True
145208,145210,548871,22162,HEART GARLAND RUSTIC PADDED,2,14410,6,True


### How often does the same item appear at both zero and a positive price?

Start with all 1,179 positive-quantity, zero-price lines on non-C invoices. For each one, check whether a positive-quantity line on that invoice records a positive price for the **same StockCode**. For an identified customer the ID must also match. For an invoice without a customer ID, the invoice and SKU establish only that the entries share a document. Count each zero-price source row once even if it has several priced counterparts. Keep `M / Manual` in a separate group because it identifies a manual entry.

### 同一商品在同票中同时出现零价和正价，有多少例？

从非 C 发票中数量为正的全部 1,179 条零价明细出发。对每一条检查同张发票上是否有**相同商品编码**、数量为正且单价为正的记录。有客户编号时还要求编号一致。缺少客户编号时，发票号和商品编码只能把记录关联到同一张单据。即使有多条正价行与其匹配，每条零价原始行也只计一次。`M / Manual` 表示手工账项，单独计数。

In [8]:
price_keys = ["InvoiceNo", "CustomerID", "StockCode"]
zero_items = lines.loc[~lines["invoice_c"] & lines["Quantity"].gt(0) & lines["UnitPrice"].eq(0)].copy()
zero_keys = zero_items[price_keys].drop_duplicates()
# Missing CustomerID matches only within the same invoice and SKU.
matching_priced = priced.merge(zero_keys, on=price_keys, how="inner", validate="many_to_one")
priced_by_item = matching_priced.groupby(price_keys, dropna=False, as_index=False).agg(
    priced_rows=("ExcelRow", "size"), priced_units=("Quantity", "sum"),
    first_priced_excel_row=("ExcelRow", "min"),
)
zero_items = zero_items.merge(priced_by_item, on=price_keys, how="left", validate="many_to_one")
zero_items["same_item_priced"] = zero_items["priced_rows"].notna()
zero_items["any_priced_line"] = zero_items["InvoiceNo"].isin(priced["InvoiceNo"])
zero_items["manual_entry"] = zero_items["StockCode"].eq("M") & zero_items["Description"].eq("Manual").fillna(False)

summary_rows = []
for (has_id, manual), group in zero_items.groupby(["customer_present", "manual_entry"]):
    any_priced = group.loc[group["any_priced_line"]]
    same_item = group.loc[group["same_item_priced"]]
    summary_rows.append({
        "customer_id_present": has_id, "manual_entry": manual, "zero_price_rows": len(group),
        "any_priced_zero_rows": len(any_priced), "any_priced_invoices": any_priced["InvoiceNo"].nunique(),
        "same_item_zero_rows": len(same_item), "same_item_invoices": same_item["InvoiceNo"].nunique(),
    })
display(pd.DataFrame(summary_rows))

,customer_id_present,manual_entry,zero_price_rows,any_priced_zero_rows,any_priced_invoices,same_item_zero_rows,same_item_invoices
0,False,False,1139,367,23,18,10
1,True,False,34,28,25,5,5
2,True,True,6,5,5,1,1


**Five identified customers each have one product line matching the pattern on invoice 550188.** Within the 40 zero-price lines with customer IDs, 33 share an invoice with a priced line. Five of the 33 are manual entries. This leaves 28 product lines on 25 invoices, of which **five lines on five invoices** match the same SKU. A sixth same-code match, on invoice 561916, has `M / Manual` on both lines.

The same-SKU check also finds 18 zero-price product lines on ten invoices without a customer ID. Across the original workbook, this gives **23 zero-price product lines on 15 invoices** with a priced counterpart for the same SKU. A zero price alongside a priced purchase is consistent with a complimentary unit. The data supply no promotion flag or reason for that price, so the actual number of gifts cannot be determined. The output lists a zero-price source row, its first priced source row and total priced units for each match.

**有客户编号的商品记录中，5 位客户各有 1 条明细符合发票 550188 的情况。** 有客户编号的 40 条零价明细中，33 条所在的发票含正价行。去掉其中 5 条手工账项，剩 28 条商品明细，涉及 25 张发票。进一步要求正价行与零价行的商品编码完全相同，则有 **5 条商品明细，分属 5 张发票**。另有一例出现在发票 561916，两行均为 `M / Manual` 手工账项。

在缺客户编号的记录中，同样的规则另外找出 18 条商品明细，涉及 10 张发票。因此全表共有 **23 条零价商品明细、15 张发票**符合“同票同商品还有正价行”。这种记录与随单赠送相符；原表没有促销标记或逐行零价原因，无法据此确定实际赠品数量。下表列出每条匹配记录的零价原始行号、首条正价原始行号和正价总件数。

In [9]:
mixed_items = zero_items.loc[zero_items["same_item_priced"]].sort_values(
    ["customer_present", "manual_entry", "InvoiceNo", "ExcelRow"], ascending=[False, True, True, True]
)
product_matches = mixed_items.loc[~mixed_items["manual_entry"]]
print(f"All matching product lines / 全表同票同商品的零价商品明细：{len(product_matches)}")
print(f"Distinct invoices / 涉及的发票：{product_matches['InvoiceNo'].nunique()}")
display(mixed_items[[
    "ExcelRow", "InvoiceNo", "CustomerID", "StockCode", "Description", "Quantity",
    "first_priced_excel_row", "priced_rows", "priced_units", "manual_entry",
]].rename(columns={"Quantity": "zero_price_units"}).astype({
    "first_priced_excel_row": "Int64", "priced_rows": "Int64", "priced_units": "Int64",
}))

All matching product lines / 全表同票同商品的零价商品明细：23
Distinct invoices / 涉及的发票：15


,ExcelRow,InvoiceNo,CustomerID,StockCode,Description,zero_price_units,first_priced_excel_row,priced_rows,priced_units,manual_entry
430,139455,548318,13113,22055,MINI CAKE STAND HANGING STRAWBERY,5,139456,1,3,False
444,145210,548871,14410,22162,HEART GARLAND RUSTIC PADDED,2,145209,1,4,False
475,157044,550188,12457,22636,CHILDS BREAKFAST SET CIRCUS PARADE,1,157031,1,8,False
1060,446796,574920,13985,23480,MINI LIGHTS WOODLAND MUSHROOMS,1,446783,1,4,False
1107,480651,577314,12444,23407,SET OF 2 TRAYS HOME SWEET HOME,2,480631,1,2,False
791,285659,561916,15581,M,Manual,1,285660,1,1,True
80,15659,537640,<NA>,21232,STRAWBERRY CERAMIC TRINKET BOX,1,15660,1,39,False
95,21051,538071,<NA>,22585,PACK OF 6 BIRDY GIFT TAGS,1,21052,1,1,False
120,30605,538877,<NA>,21479,WHITE SKULL HOT WATER BOTTLE,1,30606,1,71,False
121,30646,538877,<NA>,22112,CHOCOLATE HOT WATER BOTTLE,1,30647,1,35,False


### Invoice 550188: one SKU at two prices

Customer 12457 has two lines for SKU 22636 on the same invoice. The eight units at £7.65 yield £61.20. The other unit has a recorded price of £0.00. For a Top-10 target defined as **products purchased at a positive price**, exclude the zero-price line while keeping the eight priced units and one distinct purchased SKU in this basket. Removing the whole invoice or permanently banning the SKU would erase the priced purchase. A complimentary unit is one possible explanation. The invoice records the price but gives no reason for it.

### 发票 550188：同一商品出现两种单价

客户 12457 在同张发票上有两行商品码 22636。8 件商品按 £7.65 计价，行金额 £61.20；另 1 件的记录单价为 £0.00。若 Top-10 的目标定义为**以正价购买的商品**，则排除零价行，同时保留 8 件正常计价商品，该购物篮中这个商品编码计 1 次。删除整张发票或永久排除这个商品码都会丢失正常计价的购买记录。这与随单赠送的情形相符，发票只记录了价格，未注明赠送原因。

In [10]:
invoice_item = raw.loc[raw["InvoiceNo"].eq("550188") & raw["StockCode"].eq("22636")].copy()
invoice_item["line_gbp"] = invoice_item["Quantity"] * invoice_item["UnitPrice"]
display(invoice_item[[
    "ExcelRow", "CustomerID", "InvoiceNo", "InvoiceDate", "StockCode",
    "Description", "Quantity", "UnitPrice", "line_gbp",
]])
paid_item = invoice_item.loc[invoice_item["Quantity"].gt(0) & invoice_item["UnitPrice"].gt(0)]
display(paid_item.groupby(["CustomerID", "InvoiceNo"], as_index=False).agg(
    purchased_skus=("StockCode", "nunique"), priced_units=("Quantity", "sum"),
    priced_amount_gbp=("line_gbp", "sum")
))

,ExcelRow,CustomerID,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,line_gbp
157029,157031,12457,550188,2011-04-14 18:57:00,22636,CHILDS BREAKFAST SET CIRCUS PARADE,8,7.65,61.2
157042,157044,12457,550188,2011-04-14 18:57:00,22636,CHILDS BREAKFAST SET CIRCUS PARADE,1,0.00,0.0


,CustomerID,InvoiceNo,purchased_skus,priced_units,priced_amount_gbp
0,12457,550188,1,8,61.2


### Did the five customers pay for the same SKU on a different occasion?

For each of the five identified product matches, look for another non-C invoice on which **the same customer** bought **the same SKU** at a positive price and quantity. Count distinct invoices strictly before or after the case timestamp. The priced line on the original invoice is excluded. Dates come from the original workbook.

Four customers have an earlier qualifying invoice. None of the five have a later qualifying invoice by 9 December 2011 at 12:50. Follow-up ranges from about 21 to 254 days. These five observed histories cannot establish why a unit was free, whether it changed the customer's preferences, or what the customer bought after the workbook ended.

### 这五位客户是否在另一次交易中付费购买过同一商品？

对五条有客户编号的同票同商品案例，查找**相同客户、相同商品码**的其他非 C 发票，要求数量和单价均为正。按不同发票计数，时间必须严格早于或晚于案例行。案例发票本身的正价行排除在复购统计之外。时间取自原始工作簿。

五人中有四人此前有符合条件的付费购买。截至 2011 年 12 月 9 日 12:50，五人均无可观察到的后续同商品付费购买。各案例可观察的后续时间约为 21 至 254 天。这五段历史无法确定零价原因、零价商品对偏好的影响或观察期结束后的行为。

In [11]:
observed_until = raw["InvoiceDate"].max()
case_history = []
for case in product_matches.loc[product_matches["customer_present"]].itertuples(index=False):
    other = priced.loc[
        priced["CustomerID"].eq(case.CustomerID)
        & priced["StockCode"].eq(case.StockCode)
        & priced["InvoiceNo"].ne(case.InvoiceNo)
    ]
    prior = other.loc[other["InvoiceDate"].lt(case.InvoiceDate), "InvoiceNo"]
    later = other.loc[other["InvoiceDate"].gt(case.InvoiceDate), "InvoiceNo"]
    case_history.append({
        "ExcelRow": case.ExcelRow, "InvoiceNo": case.InvoiceNo, "CustomerID": case.CustomerID,
        "StockCode": case.StockCode, "case_time": case.InvoiceDate,
        "prior_paid_invoices": prior.nunique(), "later_paid_invoices": later.nunique(),
        "follow_up_days": round((observed_until - case.InvoiceDate) / pd.Timedelta(days=1), 1),
    })

case_history = pd.DataFrame(case_history)
display(case_history)
print("Observed through / 工作簿截止：", observed_until)
print("Cases with earlier purchases / 有更早购买的案例数：", case_history["prior_paid_invoices"].gt(0).sum(), "/", len(case_history))
print("Cases with later purchases / 有更晚购买的案例数：", case_history["later_paid_invoices"].gt(0).sum(), "/", len(case_history))

,ExcelRow,InvoiceNo,CustomerID,StockCode,case_time,prior_paid_invoices,later_paid_invoices,follow_up_days
0,139455,548318,13113,22055,2011-03-30 12:45:00,1,0,254.0
1,145210,548871,14410,22162,2011-04-04 14:42:00,1,0,248.9
2,157044,550188,12457,22636,2011-04-14 18:57:00,0,0,238.7
3,446796,574920,13985,23480,2011-11-07 16:34:00,1,0,31.8
4,480651,577314,12444,23407,2011-11-18 13:23:00,1,0,21.0


Observed through / 工作簿截止： 2011-12-09 12:50:00
Cases with earlier purchases / 有更早购买的案例数： 4 / 5
Cases with later purchases / 有更晚购买的案例数： 0 / 5


### Identify manual entries and bad-debt adjustments

The workbook uses the literal description `Manual` on 572 rows. The code counts the original, case-sensitive item codes separately: 571 use `M` and one uses `m`. Six of the 572 are among the 40 identified positive-quantity, zero-price lines. The five basic filters leave 279 priced `M` rows. A positive amount on a manual entry does not identify a product to recommend.

All three rows for `B` read `Adjust bad debt`: the signed line amounts are +£11,062.06, −£11,062.06, and −£11,062.06. All lack a customer ID. These literal labels and counts are workbook evidence. The UCI variable guide defines the C invoice prefix as a cancellation. It gives no row-level reason for a free unit, no reason field for manual entries, and no confirmation of a payment or refund. The linked project notebook applies the cleaning decisions. [UCI dataset and variable guide](https://archive.ics.uci.edu/dataset/352/online+retail) · [Project cleaning notebook](../../notebooks/project2_colab.ipynb)

### 区分手工账项和坏账调整

原始工作簿有 572 行的描述严格等于 `Manual`。以下代码按区分大小写的原始商品码计数，其中 571 行为 `M`、1 行为 `m`。在 40 条有客户编号、数量为正的零价明细中，有 6 行属于这批记录。五项基础过滤后还有 279 条正价的 `M` 行。手工账项上的正金额本身无法识别可推荐商品。

商品码 `B` 的三行原文均为 `Adjust bad debt`，带符号的行金额依次为 +£11,062.06、−£11,062.06、−£11,062.06，均缺客户编号。以上标签及计数来自原始工作簿。UCI 字段说明将 C 开头的发票定义为取消记录。工作簿未给出每件零价商品的原因、手工账项的具体缘由或款项收付凭证。正式清洗规则见下方项目主 notebook。[UCI 数据集与字段说明](https://archive.ics.uci.edu/dataset/352/online+retail) · [项目主 notebook](../../notebooks/project2_colab.ipynb)

In [12]:
manual_rows = raw.loc[raw["Description"].eq("Manual")].copy()
manual_rows["identified_zero_price"] = manual_rows["ExcelRow"].isin(identified_zero["ExcelRow"])
manual_rows["passes_basic_filters"] = (before_price & raw["UnitPrice"].gt(0)).loc[manual_rows.index]
display(manual_rows.groupby("StockCode", as_index=False).agg(
    raw_rows=("ExcelRow", "size"),
    identified_zero_price_rows=("identified_zero_price", "sum"),
    remaining_after_basic_filters=("passes_basic_filters", "sum"),
))
display(lines.loc[lines["StockCode"].eq("B"), line_fields].sort_values("ExcelRow"))

,StockCode,raw_rows,identified_zero_price_rows,remaining_after_basic_filters
0,M,571,6,279
1,m,1,0,0


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
299982,299984,A563185,2011-08-12 14:50:00,B,Adjust bad debt,1,11062.06,<NA>,11062.06
299983,299985,A563186,2011-08-12 14:51:00,B,Adjust bad debt,1,-11062.06,<NA>,-11062.06
299984,299986,A563187,2011-08-12 14:52:00,B,Adjust bad debt,1,-11062.06,<NA>,-11062.06


## Missing fields and competing filters

Read `missingness` before deciding how a field can be used. A customer ID identifies a person. Averaging customer numbers or inventing an ID would combine unrelated histories. Keep unidentified rows for this raw-data investigation and count them separately. The project excludes them when it constructs personalized purchase histories. A row with a valid StockCode can still identify a product when Description is missing. Use the code as its display label when a suitable description is unavailable. The invoice table lets you inspect which invoice numbers are affected by absent customer IDs.

The `filter_attribution` table revisits the earlier hypothetical five-rule filter in its specified order. For the 10,624 rows with nonpositive quantity, 37 are first accounted for by duplicates, 1,715 by missing customer ID, and 8,872 by the C-invoice flag. Thus quantity contributes zero additional removals at its step. The final price rule accounts for 40 additional rows, all with zero unit price. These figures describe overlap and rule order. All original rows still feed the exception tables below.

All 135,080 missing customer IDs occur on 3,710 invoices. Across the complete sheet, none of those invoices contains a known customer on another line. All 1,454 missing descriptions also lack a customer ID. Quantity and UnitPrice have no missing values. A zero price is an observed value. Lecture 3, slide 10, lists imputation as one possible response to missing data. Here a mean cannot identify a buyer or establish a paid price. [Lecture 3](../../resource/Lecture_3.pdf)

## 缺失字段与相互重叠的规则

先查看 `missingness`，再判断每个字段如何用于分析。客户编号用于识别客户，对编号取均值或虚构编号会把无关历史混在一起。本次原始数据核查保留无编号行并单独计数，项目在构建个性化购买历史时才剔除这些行。如果 StockCode 有效，即使缺少 Description 也能识别商品；没有合适的商品名称时，可直接用商品编码展示。发票表可用来定位哪些发票号涉及客户编号缺失。

`filter_attribution` 按此前假设的五规则顺序追溯净排除量。在 10,624 条非正数量行中，37 条首先归入完全重复，1,715 条首先归入客户编号缺失，8,872 条首先归入 C 发票。因此数量规则本身新增排除 0 行。最后的价格规则新增排除 40 行，均为零单价。以上是规则重叠及顺序产生的归因，后续异常表仍使用所有原始行。

缺客户编号的 135,080 行涉及 3,710 张发票。回查整张原表，这些发票的其他明细也都没有已知客户编号。缺商品描述的 1,454 行全部同时缺客户编号。数量和单价都没有空值，零价是原表实际记录的值。Lecture 3 第 10 页列出多种缺失值处理方式；在此，对编号求均值无法识别买家，用均价替代零也无法证明付费。[Lecture 3](../../resource/Lecture_3.pdf)

### Missing fields across the original worksheet · `missingness`

For each of the eight original fields, count missing rows, the percentage of all 541,909 rows they represent, and the number of distinct affected invoices. The last column counts the intersection with missing customer IDs. Of the 135,080 rows missing a customer ID, 1,454 also have a missing description. A row can appear in several field counts.

### 原始工作表的字段缺失情况 · `missingness`

针对八个原始字段分别统计缺失行数、占全部 541,909 行的比例，以及涉及的不同发票数。最后一列统计该字段和客户编号同时缺失的行数。135,080 行缺少客户编号，其中 1,454 行也缺少商品描述。一行可能计入多个字段的缺失数。

In [13]:
missing_customer = raw["CustomerID"].isna()
missing_records = []
for field in RAW_COLUMNS:
    missing = raw[field].isna()
    missing_records.append({
        "field": field,
        "raw_rows": len(raw),
        "missing_rows": int(missing.sum()),
        "missing_pct": 100 * missing.mean(),
        "invoices_with_missing": raw.loc[missing, "InvoiceNo"].nunique(),
        "missing_with_missing_customer_id": int((missing & missing_customer).sum()),
    })

tables["missingness"] = pd.DataFrame.from_records(missing_records)
display(tables["missingness"])

,field,raw_rows,missing_rows,missing_pct,invoices_with_missing,missing_with_missing_customer_id
0,InvoiceNo,541909,0,0.000000,0,0
1,StockCode,541909,0,0.000000,0,0
2,Description,541909,1454,0.268311,1454,1454
3,Quantity,541909,0,0.000000,0,0
4,InvoiceDate,541909,0,0.000000,0,0
5,UnitPrice,541909,0,0.000000,0,0
6,CustomerID,541909,135080,24.926694,3710,135080
7,Country,541909,0,0.000000,0,0


### Can another line on the same invoice supply a customer ID? · `customer_id_invoices`

Find every invoice with at least one missing customer ID. Then retrieve **all** of its lines and count the distinct known IDs. Inspect the first ten invoices and the distribution across the full table below. All 3,710 affected invoices have zero known customer IDs on their other lines. This lookup therefore supplies no customer ID to fill in. The worksheet row numbers let you check each case in Excel.

### 同一张发票的其他行能否补出客户编号？ · `customer_id_invoices`

先找出至少有一行缺少客户编号的发票，再取回这些发票的**所有行**，统计每张发票上已知的不同客户编号数。下方展示前十张发票，以及完整表中已知编号数的分布。3,710 张受影响的发票上均找不到任何已知客户编号，因此无法从同票的其他行补值。可以用工作表行号回到 Excel 核查。

In [14]:
affected_invoices = raw.loc[missing_customer, "InvoiceNo"].dropna().unique()
affected_lines = raw.loc[
    raw["InvoiceNo"].isin(affected_invoices), ["InvoiceNo", "CustomerID", "ExcelRow"]
].copy()
affected_lines["missing_customer"] = affected_lines["CustomerID"].isna()
affected_lines["missing_excel_row"] = affected_lines["ExcelRow"].where(affected_lines["missing_customer"])
missing_invoices = affected_lines.groupby("InvoiceNo", sort=False).agg(
    first_excel_row=("ExcelRow", "min"),
    first_missing_excel_row=("missing_excel_row", "min"),
    total_rows=("ExcelRow", "size"),
    missing_customer_rows=("missing_customer", "sum"),
    known_customer_ids=("CustomerID", "nunique"),
).reset_index().sort_values("first_excel_row", ignore_index=True)
missing_invoices["first_missing_excel_row"] = missing_invoices["first_missing_excel_row"].astype("int64")

tables["customer_id_invoices"] = missing_invoices
display(missing_invoices.head(10))
display(missing_invoices["known_customer_ids"].value_counts().rename("invoices").to_frame())

,InvoiceNo,first_excel_row,first_missing_excel_row,total_rows,missing_customer_rows,known_customer_ids
0,536414,624,624,1,1,0
1,536544,1445,1445,527,527,0
2,536545,1972,1972,1,1,0
3,536546,1973,1973,1,1,0
4,536547,1974,1974,1,1,0
5,536549,1989,1989,1,1,0
6,536550,1990,1990,1,1,0
7,536552,2026,2026,1,1,0
8,536553,2027,2027,1,1,0
9,536554,2028,2028,1,1,0


,invoices
known_customer_ids,
0,3710


### Which filter first accounts for each affected row? · `filter_attribution`

Apply the proposed purchase-sample rules in order to a temporary classification. Detect duplicates across the eight original fields and keep the first occurrence. Then flag a missing customer ID, a C-prefixed invoice, a nonpositive quantity, and a nonpositive unit price in that order. Each row contributes once within a scope. Scopes such as C-prefixed invoices and negative quantities can overlap. A zero under a later filter means earlier filters have already accounted for all relevant rows in that scope. All original rows remain available for inspection.

### 各类明细最先命中哪一条过滤规则？ · `filter_attribution`

按计划中的购买样本规则对原表临时分类。先基于八个原始字段判断完全重复并保留首次出现的行，再依次判断缺客户编号、发票号以 C 开头、数量非正和单价非正。一个范围内每行只计一次。不同范围之间可以重叠，例如 C 开头的发票与负数量明细。靠后的规则计数为零时，要检查前面的规则是否已经计入该范围内的所有相关行。原始明细仍可随时回查。

In [15]:
purchase_flags = {
    "exact_duplicate": raw.duplicated(subset=RAW_COLUMNS, keep="first"),
    "missing_customer_id": missing_customer,
    "cancelled_invoice": raw["InvoiceNo"].astype("string").str.startswith("C", na=False),
    "nonpositive_quantity": raw["Quantity"].le(0).fillna(False),
    "nonpositive_unit_price": raw["UnitPrice"].le(0).fillna(False),
}
first_rule = pd.Series("retained", index=raw.index)
for rule, condition in purchase_flags.items():
    first_rule.loc[first_rule.eq("retained") & condition] = rule

attribution_scopes = {
    "all_rows": pd.Series(True, index=raw.index),
    "C_invoices": purchase_flags["cancelled_invoice"],
    "Q_nonpositive": purchase_flags["nonpositive_quantity"],
    "P_nonpositive": purchase_flags["nonpositive_unit_price"],
    "zero_quantity": raw["Quantity"].eq(0),
    "negative_quantity": raw["Quantity"].lt(0),
    "zero_price": raw["UnitPrice"].eq(0),
    "negative_price": raw["UnitPrice"].lt(0),
    "missing_customer_id": missing_customer,
    "missing_description": raw["Description"].isna(),
}
attribution_rows = []
for scope, members in attribution_scopes.items():
    counts = first_rule.loc[members.fillna(False)].value_counts()
    row = {"scope": scope, "raw_rows": int(counts.sum())}
    for rule in [*purchase_flags, "retained"]:
        row[rule] = int(counts.get(rule, 0))
    attribution_rows.append(row)

tables["filter_attribution"] = pd.DataFrame.from_records(attribution_rows)
display(tables["filter_attribution"])

,scope,raw_rows,exact_duplicate,missing_customer_id,cancelled_invoice,nonpositive_quantity,nonpositive_unit_price,retained
0,all_rows,541909,5268,135037,8872,0,40,392692
1,C_invoices,9288,37,379,8872,0,0,0
2,Q_nonpositive,10624,37,1715,8872,0,0,0
3,P_nonpositive,2517,5,2472,0,0,40,0
4,zero_quantity,0,0,0,0,0,0,0
5,negative_quantity,10624,37,1715,8872,0,0,0
6,zero_price,2515,5,2470,0,0,40,0
7,negative_price,2,0,2,0,0,0,0
8,missing_customer_id,135080,43,135037,0,0,0,0
9,missing_description,1454,0,1454,0,0,0,0


## Negative quantities on non-C invoices and earlier positive quantities

Start with `nonC_negative`. The 1,336 lines each occupy a single-line invoice, have zero unit price and lack a customer ID. Of these, 862 also lack a description. Use `history_summary` to compare earlier positive-quantity lines for the exact invoice, invoice + SKU, SKU alone, and customer + SKU. Earlier means a strictly earlier timestamp. The positive-quantity lookups include every earlier row with Quantity > 0 regardless of price or missing ID. The stronger customer/SKU comparison also requires a known customer and a non-C positive-price purchase line. A blank customer match is unavailable evidence. A previous occurrence of the SKU shows the item was recorded before. Matching the customer, price, quantity magnitude, and time order provides stronger evidence of a possible reversal. Even that match alone does not verify a physical return or refund.

For the 1,336 lines, the exact-invoice and exact-invoice-plus-SKU searches each find zero earlier positive lines. The weaker SKU-only search succeeds on 1,179 lines. For the 9,288 C lines, removing the C and looking for that invoice number plus SKU finds zero earlier positive matches. A different search finds 2,816 C lines with at least one earlier non-C, positive-price line for the same customer, SKU, unit price and opposite quantity. These are candidate lines, potentially several per credit. The workbook contains no allocation of each credit to an original sale.

## 非 C 发票的负数量与此前的正数量记录

从 `nonC_negative` 入手。这 1,336 行每行各占一张仅有一行的发票，单价为零、客户编号缺失，其中 862 行还缺少商品描述。通过 `history_summary` 分别比较同发票、同发票与商品、同商品，以及同客户与商品的早先正数量记录。“早先”要求时间戳严格早于当前行。按正数量回查时，单价为零或缺客户编号的早期正数量行也参与。核对更强的同客户、同商品候选时，另要求客户编号已知，且候选购买行单价为正、发票号不以 C 开头。缺少客户编号时，客户匹配结果为空，表示无法核对。商品编码曾出现，只说明原表中此前记录过这个商品。如果客户、单价、数量绝对值和时间顺序都吻合，则更支持可能存在冲销；仅凭匹配仍无法确认实物退回或款项退回。

对 1,336 行按原发票号、以及按原发票号加商品编码查找，两个口径都没有找到此前的正数量行。放宽到仅按商品编码匹配，则有 1,179 行找到早期记录。对 9,288 条 C 明细去掉 C 前缀后再按发票号加商品回查，也没有早期匹配。换一种方法，按同一客户、同一商品、同一单价、数量恰好相反四个条件查找，2,816 行有更早的非 C、正价候选购买行。一条贷项可能对应多个候选，原表没有给出逐笔核销关系。

## Which transactions were already on record?

Use each flagged line as a query against the original worksheet. For an invoice, SKU, or customer–SKU pair, count positive-quantity **lines**, retaining repeated entries. Sort historical lines by timestamp and then by Excel row. Within each group of matching identifiers, `cumcount() + 1` assigns the number of lines seen so far. A backward `merge_asof` looks up the last count **strictly before** the query. A line with the same timestamp is ineligible, even if it occurs earlier on the worksheet.

A count of **0** means the necessary identifiers and timestamp are present and no earlier line matches. **NA** means a required identifier or timestamp is missing, or that a C-invoice diagnostic does not apply. The full-sheet invoice count includes the flagged row and all matching invoice lines, regardless of date.

## 这条明细出现之前，工作簿中已有怎样的交易记录？

用每条标记明细查询原始工作表。按发票、商品或客户与商品的组合统计数量为正的**明细行数**，保留原表中的重复行。先按交易时间排序，同一时间再按 Excel 行号排序。在标识符相同的每个组内，`cumcount() + 1` 给出截至每一行的累计行数。`merge_asof` 向前查找**严格早于**标记时间的最后一个累计值。同一时间的交易即使在表中的行号较小，也不能作为更早的记录。

计数为 **0** 表示标识符和时间齐全，但没有找到符合条件的更早明细。**NA** 表示缺少比较所需的标识符或时间，或该行不适用 C 发票专用的比较。整表发票行数包含标记行本身以及所有同发票号的行，不受时间限制。

In [16]:
def strictly_earlier_match(history, queries, keys, columns):
    right = history.dropna(subset=keys)[[*keys, "InvoiceDate", *columns]]
    left = queries.dropna(subset=keys)[[*keys, "InvoiceDate", "_position"]]
    matched = pd.merge_asof(
        left, right, on="InvoiceDate", by=keys,
        allow_exact_matches=False, direction="backward",
    )
    return matched.set_index("_position")[columns]


def earlier_line_counts(history, queries, keys):
    running = history[[*keys, "InvoiceDate"]].copy()
    running["_count"] = running.groupby(keys, sort=False).cumcount() + 1
    matched = strictly_earlier_match(running, queries, keys, ["_count"])
    return matched["_count"].fillna(0).astype("Int64")

### Compare the invoice, the item and the customer

Start with the exact invoice number, including any leading C. Then compare the invoice–SKU pair, the SKU across the whole sheet, and the customer–SKU pair. For these counts, a positive quantity is sufficient. To show a plausible purchase, narrow the search to lines with a positive price and a non-C invoice number for the same customer and SKU. Record the latest eligible time and its Excel row. If two candidates share that time, select the higher Excel row to make the example reproducible.

### 按发票、商品和客户核查

先查完整发票号（保留开头的 C），再分别按发票与商品、全表中的商品、客户与商品核查。上述计数只要求历史行数量为正。为展示一条可能的购买记录，再缩小范围：要求客户和商品均相同、单价为正、发票号存在且不以 C 开头。记录时间最近的符合条件的行。若最近时间相同，取 Excel 行号较大的一条，保证每次展示相同的示例。

In [17]:
history_fields = ["InvoiceNo", "StockCode", "CustomerID", "InvoiceDate", "Quantity", "UnitPrice"]
history_queries = flagged[history_fields].reset_index(drop=True)
full_invoice_counts = raw["InvoiceNo"].value_counts()
history_evidence = pd.DataFrame({
    "invoice_rows_full_sheet": history_queries["InvoiceNo"].map(full_invoice_counts).astype("Int64")
})
history_queries["_position"] = history_queries.index
dated_queries = history_queries.dropna(subset=["InvoiceDate"]).sort_values("InvoiceDate")
positive_history = raw.loc[
    raw["Quantity"].gt(0) & raw["InvoiceDate"].notna(), [*history_fields, "ExcelRow"]
].sort_values(["InvoiceDate", "ExcelRow"])

history_keys = {
    "earlier_positive_invoice_rows": ["InvoiceNo"],
    "earlier_positive_invoice_sku_rows": ["InvoiceNo", "StockCode"],
    "earlier_positive_sku_rows": ["StockCode"],
    "earlier_positive_customer_sku_rows": ["CustomerID", "StockCode"],
}
for field, keys in history_keys.items():
    history_evidence[field] = earlier_line_counts(positive_history, dated_queries, keys)

In [18]:
plausible_purchases = positive_history.loc[
    positive_history["UnitPrice"].gt(0)
    & positive_history["InvoiceNo"].notna()
    & ~positive_history["InvoiceNo"].str.startswith("C", na=False)
].copy()
plausible_purchases["_purchase_date"] = plausible_purchases["InvoiceDate"]
latest_purchase = strictly_earlier_match(
    plausible_purchases, dated_queries, ["CustomerID", "StockCode"],
    ["ExcelRow", "_purchase_date"],
)
history_evidence["latest_plausible_customer_sku_excel_row"] = latest_purchase["ExcelRow"].astype("Int64")
history_evidence["latest_plausible_customer_sku_date"] = latest_purchase["_purchase_date"]

### Two additional comparisons for C-prefixed invoices

First remove one leading C and count earlier positive-quantity lines with that invoice number and the same SKU. A match would show a coincidence of identifiers. Independently, look for earlier plausible purchase lines with the same customer, SKU and exact unit price whose quantity equals the cancellation quantity with its sign reversed. This second comparison can find candidates on a different invoice. A candidate helps trace the records. The worksheet alone cannot establish which invoice a credit refers to or whether goods physically came back.

### C 发票的两项补充核查

第一项只去掉发票号开头的一个 C，再统计发票号与该编号相同、商品相同且数量为正的更早明细。若有匹配，只说明标识符吻合。第二项独立比较客户、商品、单价和数量：更早的可能购买记录必须有完全相同的客户、商品和单价，购买数量必须等于冲销行数量的相反数。这项核查可以找到不同发票上的候选行，供回查原始明细。仅凭这些字段无法确认冲销所指的原发票或货物是否实际退回。

In [19]:
c_queries = dated_queries.loc[dated_queries["InvoiceNo"].str.startswith("C", na=False)].copy()
c_queries["InvoiceNo"] = c_queries["InvoiceNo"].str.slice(1).replace("", pd.NA)
history_evidence["earlier_positive_stripped_invoice_sku_rows"] = earlier_line_counts(
    positive_history, c_queries, ["InvoiceNo", "StockCode"],
)
c_queries["Quantity"] = -c_queries["Quantity"]
history_evidence["earlier_matching_customer_sku_price_opposite_quantity_rows"] = earlier_line_counts(
    plausible_purchases, c_queries, ["CustomerID", "StockCode", "UnitPrice", "Quantity"],
)

# ExcelRow identifies the same source line across reruns. / Excel 行号始终指向同一条原始明细。
evidence_by_row = history_evidence.set_axis(flagged["ExcelRow"])
flagged = flagged.drop(columns=history_evidence.columns, errors="ignore").join(
    evidence_by_row, on="ExcelRow", validate="one_to_one",
)

## Annotate the lines with literal evidence

The following rules retain the words that match and the exact special code. A generic product name usually has no matching signal. The original description stays alongside each annotation.

## 给明细标记可回查的字面证据

下方规则记录命中的原始词语和特殊编码。普通商品名称常常没有关键词命中；每个标记旁都保留原始描述供复查。

In [20]:
# Keep the text exactly as written for the evidence column. Word boundaries prevent
# matches such as "count" in "country". A standalone "check" is a note, whereas
# "BROWN CHECK CAT DOORSTOP" describes a product.
_NOTE_RULES = {
    "damage_or_disposal": r"\b(?:damages?|damaged|breakages?|broken|faulty|smashed|crushed|cracked|destroyed|unsaleable|throw(?:n)? away|mouldy)\b",
    "missing_or_lost": r"\b(?:missing|lost|can't find|not rcvd)\b",
    "stock_adjustment_or_count": (
        r"\b(?:adjust(?:ment)?(?!\s+bad debt)|counted|stock check|incorrect stock entry|found)\b"
        r"|^\s*check(?=[\s?.!]*$)"
    ),
    "set_or_unit_sale_wording": (
        r"\bsold (?:as (?:sets?|1)|sets?)(?:/\d+)?"
        r"(?: (?:on|by) (?:dotcom|amazon)(?: and amazon)?)?\b"
    ),
    "item_code_or_label_discrepancy": (
        r"\b(?:(?:code|label) mix[- ]up|wrong(?:ly)? (?:coded|marked|mrked|code|barcode)"
        r"|came coded as|barcode problem)\b"
    ),
    "sample_or_promotion": r"\b(?:samples?|promotion(?:al)?|given away|free sample)\b",
    "fee_or_charge": r"\b(?:postage|carriage|bank charges|amazon fee|CRUK commission)\b",
    "discount": r"\bdiscount\b",
    "manual": r"^\s*manual\s*$",
    "debt": r"\bbad debt\b",
    "credit_note_wording": r"\b(?:credit|credits|credited|creditted)\b",
}
_NOTE_PATTERNS = {key: re.compile(expr, re.IGNORECASE) for key, expr in _NOTE_RULES.items()}
# These exact, case-sensitive codes have corresponding workbook descriptions.
_SPECIAL_CODES = {
    "S": "sample", "D": "discount", "B": "bad_debt", "M": "manual", "m": "manual",
    "POST": "postage", "DOT": "dotcom_postage", "C2": "carriage",
    "AMAZONFEE": "amazon_fee", "BANK CHARGES": "bank_charges", "CRUK": "commission",
}
# Workbook invoice identifiers have six digits, sometimes with a literal C or A
# prefix. Short numbers (e.g. "see 47") and longer alphanumeric IDs are not links.
_INVOICE_TOKEN = re.compile(r"\b[AC]?[0-9]{6}\b")

In [21]:
def _note_matches(text):
    groups, terms = [], []
    for name, pattern in _NOTE_PATTERNS.items():
        matches = pattern.findall(text)
        if matches:
            groups.append(name)
            terms.extend(match.strip() for match in matches)
    status = "blank" if not text.strip() else "unknown"
    return "|".join(groups) or status, "|".join(dict.fromkeys(terms))


def _mentioned_invoices(text):
    return list(dict.fromkeys(_INVOICE_TOKEN.findall(text)))

def description_evidence(raw: pd.DataFrame, flagged: pd.DataFrame) -> pd.DataFrame:
    """Return four evidence strings on flagged.index in the same order.

    Look up the original Description and StockCode by unique worksheet ExcelRow.
    Multiple signals, exact matched substrings, or distinct invoice tokens are
    joined by '|'. Literal text matches ignore case. Special codes match exactly.
    'blank' means missing/whitespace text. 'unknown' means no recognized text
    pattern; this also covers ordinary product names. An empty code signal or
    invoice list means no match. A note or code never establishes a return.
    """
    rows = raw[["ExcelRow", "Description", "StockCode"]].set_index("ExcelRow").loc[flagged["ExcelRow"]]
    notes = rows["Description"].astype("string").fillna("")
    matches = {note: _note_matches(note) for note in notes.unique()}
    references = {note: "|".join(_mentioned_invoices(note)) for note in matches}
    result = pd.DataFrame({
        "description_note_signals": notes.map({k: v[0] for k, v in matches.items()}),
        "description_matched_terms": notes.map({k: v[1] for k, v in matches.items()}),
        "special_code_signal": rows["StockCode"].map(_SPECIAL_CODES).fillna(""),
        "mentioned_invoice_ids": notes.map(references),
    })
    result.index = flagged.index
    return result

flagged = flagged.join(description_evidence(raw, flagged))

## Inspect the three sets side by side

The code prints the full size of each set and displays the first six rows in worksheet order. They are examples, not a statistical sample. `ExcelRow` points to the original worksheet including the header. A C-prefix is the dataset's cancellation indicator. A negative quantity records a decrease and a zero price gives a zero recorded amount regardless of quantity. Read the line description and invoice context before attaching a reason. Look at zero and negative values separately. In this workbook no quantity equals zero, 2,515 rows have zero unit price and two have negative unit price. The full sets, with additional evidence columns, are written to Excel.

## 并列检查三组记录

代码打印各组完整行数，并按工作表顺序展示前六行，便于查看实例。这里采用顺序展示，不能据此估计比例。`ExcelRow` 对应原始工作表的行号，首行为表头。C 前缀是数据集的取消标记。负数量记录数量减少，单价为零时，无论数量多少，行金额均为零。解释原因时还要查看描述与同票记录。数量、单价的零值和负值分别统计。该工作簿中数量为零有 0 行，单价为零有 2,515 行，单价为负有 2 行。完整记录及更多证据列均写入 Excel。

In [22]:
views = {
    "C_invoices": flagged.loc[flagged.invoice_c],
    "Q_nonpositive": flagged.loc[flagged.Quantity.le(0)],
    "P_nonpositive": flagged.loc[flagged.UnitPrice.le(0)],
    "nonC_negative": flagged.loc[~flagged.invoice_c & flagged.Quantity.lt(0)],
}
tables.update(views)
for name, group in views.items():
    print(f"{name} · {len(group):,} rows / 行")
    display(group[line_fields].head(6))

C_invoices · 9,288 rows / 行


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
141,143,C536379,2010-12-01 09:41:00,D,Discount,-1,27.50,14527,-27.50
154,156,C536383,2010-12-01 09:49:00,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65,15311,-4.65
235,237,C536391,2010-12-01 10:24:00,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65,17548,-19.80
236,238,C536391,2010-12-01 10:24:00,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29,17548,-6.96
237,239,C536391,2010-12-01 10:24:00,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29,17548,-6.96
238,240,C536391,2010-12-01 10:24:00,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29,17548,-6.96


Q_nonpositive · 10,624 rows / 行


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
141,143,C536379,2010-12-01 09:41:00,D,Discount,-1,27.50,14527,-27.50
154,156,C536383,2010-12-01 09:49:00,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,4.65,15311,-4.65
235,237,C536391,2010-12-01 10:24:00,22556,PLASTERS IN TIN CIRCUS PARADE,-12,1.65,17548,-19.80
236,238,C536391,2010-12-01 10:24:00,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,0.29,17548,-6.96
237,239,C536391,2010-12-01 10:24:00,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,0.29,17548,-6.96
238,240,C536391,2010-12-01 10:24:00,21980,PACK OF 12 RED RETROSPOT TISSUES,-24,0.29,17548,-6.96


P_nonpositive · 2,517 rows / 行


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
622,624,536414,2010-12-01 11:52:00,22139,NaN,56,0.0,<NA>,0.0
1970,1972,536545,2010-12-01 14:32:00,21134,NaN,1,0.0,<NA>,0.0
1971,1973,536546,2010-12-01 14:33:00,22145,NaN,1,0.0,<NA>,0.0
1972,1974,536547,2010-12-01 14:33:00,37509,NaN,1,0.0,<NA>,0.0
1987,1989,536549,2010-12-01 14:34:00,85226A,NaN,1,0.0,<NA>,0.0
1988,1990,536550,2010-12-01 14:34:00,85044,NaN,1,0.0,<NA>,0.0


nonC_negative · 1,336 rows / 行


,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
2406,2408,536589,2010-12-01 16:50:00,21777,NaN,-10,0.0,<NA>,-0.0
4347,4349,536764,2010-12-02 14:42:00,84952C,NaN,-38,0.0,<NA>,-0.0
7188,7190,536996,2010-12-03 15:30:00,22712,NaN,-20,0.0,<NA>,-0.0
7189,7191,536997,2010-12-03 15:30:00,22028,NaN,-20,0.0,<NA>,-0.0
7190,7192,536998,2010-12-03 15:30:00,85067,NaN,-6,0.0,<NA>,-0.0
7192,7194,537000,2010-12-03 15:32:00,21414,NaN,-22,0.0,<NA>,-0.0


**Reading the output**

**`C_invoices`** · All 9,288 lines on the 3,836 C-prefixed invoices have negative quantities and positive unit prices. They include 77 discount lines (StockCode D), so the C prefix alone cannot establish that a customer physically returned an item.

**`Q_nonpositive`** · All 10,624 lines with Quantity ≤ 0 are strictly negative. They split exactly into 9,288 C-invoice lines and 1,336 non-C lines.

**`P_nonpositive`** · Of 2,517 lines with UnitPrice ≤ 0, 2,515 have zero recorded price. The two negative-price lines are A563186 and A563187, each with quantity 1, unit price −£11,062.06 and the description “Adjust bad debt”.

**`nonC_negative`** · All 1,336 non-C negative-quantity lines are on single-line invoices with zero unit price and missing customer ID. An earlier positive-quantity line exists for the same SKU in 1,179 cases, but there is no customer ID to link it to a buyer.

**如何解读输出**

**`C_invoices`** · 3,836 张 C 发票的 9,288 行均为负数量、正单价，其中包括 77 行折扣记录（StockCode 为 D）。仅凭 C 前缀无法认定客户实际退回了商品。

**`Q_nonpositive`** · Quantity ≤ 0 的 10,624 行全部为负数。按发票前缀划分，恰好是 9,288 行 C 发票记录和 1,336 行非 C 记录。

**`P_nonpositive`** · UnitPrice ≤ 0 的 2,517 行中，2,515 行记录的单价为零。仅 A563186 和 A563187 两行单价为负，两行均为数量 1、单价 −£11,062.06，描述为“Adjust bad debt”（坏账调整）。

**`nonC_negative`** · 全部 1,336 行非 C 负数量记录均为单行发票，单价为零且没有客户编号。其中 1,179 行的商品能找到较早的正数量记录，但缺少客户编号，无法据此关联买家。

## Summarize historical matches and note wording

Build `history_summary` and `note_summary` from the groups below. One source row can appear in several scopes. Count matching rows and show the combinations of literal note and stock-code signals.

## 按筛选范围汇总历史匹配与备注

按下方的核查范围计算 `history_summary` 和 `note_summary`，同一原始行可以进入多个范围。分别统计有历史匹配的明细行数，以及各种备注线索与商品码的组合。

In [23]:
scopes = {
    "any_exception": flagged,
    "C_invoices": views["C_invoices"],
    "nonC_negative": views["nonC_negative"],
    "zero_qty": flagged.loc[flagged.Quantity.eq(0)],
    "positive_qty_zero_price": flagged.loc[flagged.Quantity.gt(0) & flagged.UnitPrice.eq(0)],
    "negative_price": flagged.loc[flagged.UnitPrice.lt(0)],
}

In [24]:
history_rows = []
for scope, group in scopes.items():
    history_rows.append({
        "cohort": scope, "rows": len(group), "distinct_invoices": group.InvoiceNo.nunique(),
        "singleton_invoice_rows": group.invoice_rows_full_sheet.eq(1).sum(),
        "prior_same_invoice_rows_matched": group.earlier_positive_invoice_rows.gt(0).sum(),
        "prior_same_invoice_sku_rows_matched": group.earlier_positive_invoice_sku_rows.gt(0).sum(),
        "prior_sku_rows_matched": group.earlier_positive_sku_rows.gt(0).sum(),
        "customer_sku_comparable_rows": group.earlier_positive_customer_sku_rows.notna().sum(),
        "prior_customer_sku_rows_matched": group.earlier_positive_customer_sku_rows.gt(0).sum(),
        "prior_stripped_invoice_sku_rows_matched": group.earlier_positive_stripped_invoice_sku_rows.gt(0).sum(),
        "prior_customer_sku_price_opposite_units_rows_matched":
            group.earlier_matching_customer_sku_price_opposite_quantity_rows.gt(0).sum(),
        "same_invoice_has_positive_priced_rows": group.same_invoice_positive_priced_rows.gt(0).sum(),
    })
tables["history_summary"] = pd.DataFrame(history_rows)
display(tables["history_summary"])

,cohort,rows,distinct_invoices,singleton_invoice_rows,prior_same_invoice_rows_matched,prior_same_invoice_sku_rows_matched,prior_sku_rows_matched,customer_sku_comparable_rows,prior_customer_sku_rows_matched,prior_stripped_invoice_sku_rows_matched,prior_customer_sku_price_opposite_units_rows_matched,same_invoice_has_positive_priced_rows
0,any_exception,11805,5993,4229,0,0,11391,8945,7627,0,2816,400
1,C_invoices,9288,3836,2131,0,0,9102,8905,7599,0,2816,0
2,nonC_negative,1336,1336,1336,0,0,1179,0,0,0,0,0
3,zero_qty,0,0,0,0,0,0,0,0,0,0,0
4,positive_qty_zero_price,1179,819,760,0,0,1108,40,28,0,0,400
5,negative_price,2,2,2,0,0,2,0,0,0,0,0


In [25]:
note_tables = []
for scope, group in scopes.items():
    counts = group.groupby(["description_note_signals", "special_code_signal"],
                           dropna=False).size().rename("rows").reset_index()
    counts.insert(0, "scope", scope)
    note_tables.append(counts.sort_values("rows", ascending=False, kind="stable"))
tables["note_summary"] = pd.concat(note_tables, ignore_index=True)
display(tables["note_summary"].loc[tables["note_summary"]["scope"].eq("nonC_negative")])

,scope,description_note_signals,special_code_signal,rows
38,nonC_negative,blank,,862
39,nonC_negative,damage_or_disposal,,171
40,nonC_negative,stock_adjustment_or_count,,141
41,nonC_negative,unknown,,88
42,nonC_negative,set_or_unit_sale_wording,,32
43,nonC_negative,missing_or_lost,,18
44,nonC_negative,item_code_or_label_discrepancy,,14
45,nonC_negative,credit_note_wording,,3
46,nonC_negative,sample_or_promotion,,3
47,nonC_negative,damage_or_disposal|sample_or_promotion,,2


**Reading the output**

**`history_summary`** · Of 8,905 C-invoice lines with a known customer, 7,599 have an earlier positive-quantity line for the same customer and SKU. For 2,816 C lines, the earlier unit price and opposite quantity also match. The original purchase invoice is still unconfirmed.

**`note_summary`** · Of the 1,336 non-C negative-quantity lines, 862 have blank descriptions. Among the exclusive note combinations, 171 say damage or disposal only and 141 say stock adjustment or count only.

**如何解读输出**

**`history_summary`** · 在 8,905 行有客户编号的 C 发票记录中，7,599 行能找到同一客户、同一商品的较早正数量记录。其中 2,816 行还匹配相同单价及数值相等、符号相反的数量，原购买发票仍待核实。

**`note_summary`** · 1,336 行非 C 负数量记录中，862 行描述为空。按互斥的备注线索组合统计，171 行只命中损坏或报废措辞，141 行只命中库存调整或盘点措辞。

## Read the description, item code, and invoice reference

The note summary counts literal text and special-code signals by group. `blank` means there was no description to interpret; `unknown` means the text did not match a recognized note. Inspect a note and its referenced document as separate pieces of evidence. For instance, invoices `554550` and `554553` cite `C550456` and contain “incorrectly credited”. Invoice `554551` says “reverse previous adjustment”. The signs and text support an adjustment sequence. The workbook does not identify a corresponding shipment. For `BATHROOM METAL SIGN`, the two raw descriptions differ in trailing whitespace. The name lookup trims whitespace only to find the example and retains both stock codes. Similar descriptions cannot establish an SKU alias.

For example, the description for invoice 554143 says “damages”, while 573482 says only “check”. The first is evidence of a damage notation. The second leaves the result of that check unknown. On the zero-price rows, 1,179 have positive quantity. Among those, 40 have a customer ID and 1,139 do not. Distinguishing gifts from stock entries or a missing charge requires further documentation.

## 核查原始描述、商品编码和备注引用的发票

备注汇总按组统计字面文字和特殊商品码信号。`blank` 表示没有描述可供解释；`unknown` 表示文本未命中已设置的备注规则。备注内容和它指向的单据可以分别核查。例如 `554550` 与 `554553` 的描述包含 “incorrectly credited”，并提及 `C550456`；`554551` 则写有 “reverse previous adjustment”。正负号和描述共同支持存在调整的解释，表内没有标明对应的发货记录。`BATHROOM METAL SIGN` 的两条原始描述在末尾空格上存在差别。名称查找只为定位示例去除首尾空格，同时保留两个商品编码。仅凭名称相似无法确定两个编码互为别名。

例如，发票 554143 备注为 “damages”，说明经办记录提到了损坏；573482 只写 “check”，既未说明盘点结果也未说明客户退货。零价行中有 1,179 行数量为正，其中 40 行有客户编号、1,139 行无客户编号。赠品、库存调整或漏记价格等可能原因仍需其他凭证。

In [26]:
_TRACE_COLUMNS = [
    "ExcelRow", "InvoiceNo", "StockCode", "Description", "mentioned_invoice_id",
    "referenced_invoice_rows", "earlier_positive_quantity_rows",
    "earlier_positive_quantity_sku_rows", "earlier_same_sku_rows",
    "earlier_credit_or_nonpositive_rows", "earlier_credit_or_nonpositive_sku_rows",
    "example_earlier_same_sku_excel_row",
]


def mentioned_invoice_trace(raw: pd.DataFrame, flagged: pd.DataFrame) -> pd.DataFrame:
    """Trace each distinct invoice-shaped token per flagged source row, including misses.

    Match the full literal document ID in the whole input (prefix and zeros
    included). referenced_invoice_rows counts all matching raw lines. All other
    counts use strictly earlier InvoiceDate. A positive-quantity line can have a
    nonpositive price. The credit-or-nonpositive signal means a C-prefixed invoice,
    Quantity <= 0, or UnitPrice <= 0. Counts can overlap. Missing source dates or
    required SKU keys yield NA. The example is the lowest ExcelRow among strictly
    earlier exact invoice+SKU matches. An invoice match alone never links a return.
    """
    descriptions = flagged["Description"].astype("string").fillna("")
    extracted = descriptions.map({note: _mentioned_invoices(note) for note in descriptions.unique()})
    mentioned = flagged.loc[extracted.str.len().gt(0), [
        "ExcelRow", "InvoiceNo", "StockCode", "Description", "InvoiceDate",
    ]].copy()
    mentioned["mentioned_invoice_id"] = extracted[extracted.str.len().gt(0)]
    mentioned = mentioned.explode("mentioned_invoice_id")
    target_rows = raw.loc[raw["InvoiceNo"].isin(mentioned["mentioned_invoice_id"])]
    grouped = {key: group for key, group in target_rows.groupby("InvoiceNo", sort=False)}
    empty = target_rows.iloc[:0]
    records = []
    for row in mentioned.itertuples(index=False):
        lines = grouped.get(row.mentioned_invoice_id, empty)
        record = {
            "ExcelRow": row.ExcelRow, "InvoiceNo": row.InvoiceNo, "StockCode": row.StockCode,
            "Description": row.Description, "mentioned_invoice_id": row.mentioned_invoice_id,
            "referenced_invoice_rows": len(lines),
        }
        if pd.notna(row.InvoiceDate):
            earlier = lines.loc[lines["InvoiceDate"].lt(row.InvoiceDate)]
            positive = earlier["Quantity"].gt(0)
            credit_or_nonpositive = (
                earlier["InvoiceNo"].str.startswith("C", na=False)
                | earlier["Quantity"].le(0) | earlier["UnitPrice"].le(0)
            )
            record["earlier_positive_quantity_rows"] = int(positive.sum())
            record["earlier_credit_or_nonpositive_rows"] = int(credit_or_nonpositive.sum())
            if pd.notna(row.StockCode):
                sku = earlier["StockCode"].eq(row.StockCode)
                record["earlier_same_sku_rows"] = int(sku.sum())
                record["earlier_positive_quantity_sku_rows"] = int((positive & sku).sum())
                record["earlier_credit_or_nonpositive_sku_rows"] = int((credit_or_nonpositive & sku).sum())
                record["example_earlier_same_sku_excel_row"] = earlier.loc[sku, "ExcelRow"].min()
        records.append(record)
    trace = pd.DataFrame.from_records(records, columns=_TRACE_COLUMNS)
    return trace.astype({col: "Int64" for col in _TRACE_COLUMNS[5:]})
tables["invoice_references"] = mentioned_invoice_trace(raw, flagged)
display(tables["invoice_references"])

,ExcelRow,InvoiceNo,StockCode,Description,mentioned_invoice_id,referenced_invoice_rows,earlier_positive_quantity_rows,earlier_positive_quantity_sku_rows,earlier_same_sku_rows,earlier_credit_or_nonpositive_rows,earlier_credit_or_nonpositive_sku_rows,example_earlier_same_sku_excel_row
0,203753,554550,47566B,incorrectly credited C550456 see 47,C550456,5,0,0,1,5,1,160144
1,203763,554553,47566B,incorrectly credited C550456 see 47,C550456,5,0,0,1,5,1,160144


In [27]:
def sku_name_evidence(raw: pd.DataFrame, name: str = "BATHROOM METAL SIGN") -> pd.DataFrame:
    """One row per exact raw (Description, StockCode) in a case-sensitive trimmed lookup.

    Trimming only selects examples. Grouping retains original label whitespace,
    capitalization and exact SKU. Counts include all workbook lines. first_excel_row
    lets the reader inspect each original label and source SKU on the worksheet.
    No item map is changed.
    """
    descriptions = raw["Description"].astype("string")
    rows = raw.loc[descriptions.str.strip().eq(name.strip()).fillna(False)]
    result = (
        rows.groupby(["Description", "StockCode"], sort=True, dropna=False)
        .agg(workbook_rows=("ExcelRow", "size"), first_excel_row=("ExcelRow", "min"))
        .reset_index()
    )
    result["exact_description_length"] = result["Description"].astype("string").str.len()
    result["distinct_skus_for_exact_description"] = (
        result.groupby("Description", dropna=False)["StockCode"].transform("nunique")
    )
    result["distinct_skus_for_trimmed_name"] = rows["StockCode"].nunique()
    return result
tables["sku_names"] = sku_name_evidence(raw)
display(tables["sku_names"])

,Description,StockCode,workbook_rows,first_excel_row,exact_description_length,distinct_skus_for_exact_description,distinct_skus_for_trimmed_name
0,BATHROOM METAL SIGN,82580,641,343,19,1,2
1,BATHROOM METAL SIGN,21171,73,13319,20,1,2


**Reading the output**

**`invoice_references`** · The notes in rows 203,753 and 203,763 explicitly name C550456. Of its five earlier credit lines, row 160,144 is the sole SKU match (47566B). The reference leads to a credit document.

**`sku_names`** · Trimming whitespace makes two labels read “BATHROOM METAL SIGN”. SKU 82580 has 641 lines with a 19-character label and SKU 21171 has 73 lines with a trailing space. Keep the codes separate pending item-master evidence.

**如何解读输出**

**`invoice_references`** · 原表第 203,753 和 203,763 行的备注明确写出 C550456。该贷项发票的五行较早记录中，仅第 160,144 行的商品编码 47566B 匹配，备注追溯到的是一张贷项单据。

**`sku_names`** · 去除首尾空格后，两种标签均显示为“BATHROOM METAL SIGN”。编码 82580 有 641 行，原始标签长 19 个字符；编码 21171 有 73 行，末尾多一个空格。在获得商品主数据证据前分别保留这两个编码。

## Compare the named cases against their source rows · `case_examples`

- **A large reversal.** Compare `541431` and `C541433`. The same customer, SKU and unit price appear 16 minutes apart with quantities +74,215 and −74,215. The amounts offset. The two invoice IDs differ, even after removing the C.
- **A discount and an unpriced item.** On `C536379`, SKU `D` has the description “Discount”. A credit can refer to a discount. On `539722`, SKU `22423` has positive quantity and zero price alongside priced items. The recorded zero leaves the reason open, including whether the item was free.
- **Ledger and stock adjustments.** The code `BANK CHARGES` identifies a bank-charge entry. Compare the positive and negative prices of the `B` entries `A563185`–`A563187` (“Adjust bad debt”). These fee and debt records require different interpretations from a merchandise return. Then follow SKU `47566B` through `C550456`, `554550`, `554551`, and `554553`. Invoice `560039` explicitly mentions another code in the box; use the recorded SKU when tracing this row.

The tables show what the workbook records. Customer intent, the cause of a price of zero, physical stock movement, and settlement may require other records.

## 将指定案例与原始行对照 · `case_examples`

- **大额正负对冲。** 比较 `541431` 与 `C541433`：客户、商品编码、单价相同，相隔 16 分钟，数量分别为 +74,215 和 −74,215，两个行金额相抵。即使去掉 C，发票号也不同。
- **折扣与零价商品。** `C536379` 的商品码为 `D`，描述为 “Discount”（折扣），说明贷项可以涉及折扣。`539722` 同票中还有正常计价行，而 `22423` 数量为正、单价为零；原表留下了零价记录，是否赠品等具体原因仍待核查。
- **账务与库存调整。** 商品码 `BANK CHARGES` 标识银行费用。对照 `A563185` 至 `A563187` 的 `B` 行，查看 “Adjust bad debt”（坏账调整）所对应的正负单价。核查这类费用和坏账时应分别考虑其账务含义，不能据负号把它们当成商品退货。再沿 `47566B` 追踪 `C550456`、`554550`、`554551` 和 `554553`。`560039` 的描述明确提到箱中另一个商品编码；追踪本行时须保留原始商品编码。

表格给出工作簿记载的事实。客户意图、零价的具体原因、实际库存流向和款项结算可能需要其他单据印证。

In [28]:
examples = []
for case, invoices, sku in [
    ("same-customer opposite entries", ["541431", "C541433"], "23166"),
    ("discount credit", ["C536379"], "D"),
    ("zero recorded unit price", ["539722"], "22423"),
    ("same invoice and SKU at two prices", ["550188"], "22636"),
    ("bad-debt entries", ["A563185", "A563186", "A563187"], "B"),
    ("explicit credit note and later adjustments", ["C550456", "554550", "554551", "554553"], "47566B"),
    ("stock-code discrepancy in description", ["560039"], "20713"),
    ("damage wording in description", ["554143"], "79030G"),
    ("count/check wording in description", ["573482"], "21181"),
]:
    sample = lines.loc[lines.InvoiceNo.isin(invoices) & lines.StockCode.eq(sku)].copy()
    sample.insert(0, "case", case)
    examples.append(sample)
tables["case_examples"] = pd.concat(examples, ignore_index=True)
display(tables["case_examples"][["case", *line_fields]])
case_invoice = raw.loc[raw["InvoiceNo"].eq("539722")]
display(case_invoice.groupby("UnitPrice", as_index=False).agg(
    rows=("ExcelRow", "size"), units=("Quantity", "sum")
))
bank_fees = raw.loc[raw["StockCode"].eq("BANK CHARGES")]
display(bank_fees[["ExcelRow", "InvoiceNo", "StockCode", "Description", "Quantity", "UnitPrice"]].head(4))

,case,ExcelRow,InvoiceNo,InvoiceDate,StockCode,Description,Quantity,UnitPrice,CustomerID,amount_gbp
0,same-customer opposite entries,61621,541431,2011-01-18 10:01:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,74215,1.04,12346,77183.60
1,same-customer opposite entries,61626,C541433,2011-01-18 10:17:00,23166,MEDIUM CERAMIC TOP STORAGE JAR,-74215,1.04,12346,-77183.60
2,discount credit,143,C536379,2010-12-01 09:41:00,D,Discount,-1,27.50,14527,-27.50
3,zero recorded unit price,40091,539722,2010-12-21 13:45:00,22423,REGENCY CAKESTAND 3 TIER,10,0.00,14911,0.00
4,same invoice and SKU at two prices,157031,550188,2011-04-14 18:57:00,22636,CHILDS BREAKFAST SET CIRCUS PARADE,8,7.65,12457,61.20
5,same invoice and SKU at two prices,157044,550188,2011-04-14 18:57:00,22636,CHILDS BREAKFAST SET CIRCUS PARADE,1,0.00,12457,0.00
6,bad-debt entries,299984,A563185,2011-08-12 14:50:00,B,Adjust bad debt,1,11062.06,<NA>,11062.06
7,bad-debt entries,299985,A563186,2011-08-12 14:51:00,B,Adjust bad debt,1,-11062.06,<NA>,-11062.06
8,bad-debt entries,299986,A563187,2011-08-12 14:52:00,B,Adjust bad debt,1,-11062.06,<NA>,-11062.06
9,explicit credit note and later adjustments,160144,C550456,2011-04-18 13:08:00,47566B,TEA TIME PARTY BUNTING,-1300,2.55,15749,-3315.00


,UnitPrice,rows,units
0,0.00,1,10
1,0.55,2,120
2,0.85,1,24
3,1.65,2,22
4,1.95,1,10
5,2.10,4,66
6,2.95,5,48
7,3.75,1,8
8,4.25,1,6
9,4.65,1,8


,ExcelRow,InvoiceNo,StockCode,Description,Quantity,UnitPrice
4406,4408,536779,BANK CHARGES,Bank Charges,1,15.00
14435,14437,C537572,BANK CHARGES,Bank Charges,-1,95.38
28992,28994,C538680,BANK CHARGES,Bank Charges,-1,966.92
62508,62510,541505,BANK CHARGES,Bank Charges,1,15.00


## Amounts and customer activity under two definitions

Compare all original lines with lines on non-C invoices having positive quantity and positive unit price. The latter is a descriptive baseline for this calculation. It may still contain fees or other non-merchandise codes. The first table separates identified and unidentified customers and reports positive, negative, and signed line amounts. The second groups identified customers by whether they have regular entries, exceptions, or both. Recency is measured from the day after the latest workbook date. Compare days since **any activity** with days since a **regular purchase**. The signed amount is an arithmetic result, not evidence that payment was collected.

Chen, Sain and Guo study calendar 2011 and group UK transactions by postcode. They compute a line amount as quantity times price and isolate unusual customer profiles for separate examination before clustering (pp. 199–203). Our eight-column workbook has no postcode. We adapt their recency, distinct-invoice frequency and monetary measures to observed CustomerID over the workbook window and compare two definitions of activity. Among 1,563 customers with both kinds of lines, the median recency is 30 days for regular lines and 26 days when exceptional entries count as activity. Another 34 customers have exceptional entries only. These figures describe observed activity and cannot be used as cutoff-safe features for training. The paper supplies no transaction-level return or zero-price label.

## 两种统计口径下的金额和客户活动

对比全部原始行与“非 C 发票、数量为正且单价为正”的行。后者用作本次比较基准，其中仍可能含有费用或其他非商品码。第一张表区分客户编号是否缺失，分别统计正、负行金额及带符号金额合计。第二张表按是否有常规记录、异常记录或两者都有，对可识别客户分组。距最近活动天数均以工作簿最后一个日期的次日为基准，分别计算距**任意活动**和距**常规购买**的天数。带符号金额来自数量乘以单价，实际收款情况需另行核实。

Chen、Sain 与 Guo 研究 2011 自然年的英国交易，按邮编汇总，以数量乘单价计算行金额，并将异常客户画像单独审视后再聚类（第 199–203 页）。我们的八列表没有邮编，因此改用已有 CustomerID，按同一数据窗口比较距最近记录的天数、不同发票数和金额。1,563 位同时有两类记录的客户中，只看常规行时距最近记录天数的中位数为 30 天，计入特殊行后为 26 天。另有 34 位客户只出现于特殊行。这是全期描述；训练特征仍需按各自时间截点重算。论文没有为单笔记录提供退货或零价成因标签。

### How much signed value is attached to each type of line? · `amount_by_record_type`

Compute quantity × unit price in GBP on every original row, including duplicates. An exception line has a C-prefixed invoice, nonpositive quantity or nonpositive unit price. Split both regular and exception lines by whether the customer ID is known. The four row counts and signed totals add up to the whole worksheet. An invoice can contain more than one type of line, so adding the distinct-invoice counts across groups may count it twice. The negative and positive columns show how the signed total is formed. The data record no cash settlement or confirmation of a physical return.

### 不同类型明细分别涉及多少带符号金额？ · `amount_by_record_type`

保留原表全部明细（包括重复行），逐行计算数量 × 单价，单位为英镑。异常行指发票号以 C 开头、数量非正或单价非正的行。常规行和异常行再各自按有无客户编号拆分。四组的行数和带符号金额可以加总回原表。由于一张发票可能同时含多类行，直接跨组相加不同发票数可能重复计数。正、负金额列分别显示带符号总额的组成。原表没有实际资金结算流水，也没有实物退货确认记录。

In [29]:
impact_lines = raw[["CustomerID", "InvoiceNo", "InvoiceDate"]].copy()
impact_lines["signed_gbp"] = raw["Quantity"] * raw["UnitPrice"]
impact_lines["positive_line_gbp"] = impact_lines["signed_gbp"].clip(lower=0)
impact_lines["negative_line_gbp"] = impact_lines["signed_gbp"].clip(upper=0)
impact_lines["record_type"] = pd.Categorical.from_codes(
    exception.astype(int), categories=["regular", "exception"],
)
impact_lines["customer_id_status"] = pd.Categorical.from_codes(
    impact_lines["CustomerID"].isna().astype(int), categories=["identified", "missing"],
)

tables["amount_by_record_type"] = impact_lines.groupby(
    ["record_type", "customer_id_status"], observed=False,
).agg(
    rows=("InvoiceNo", "size"),
    distinct_invoices=("InvoiceNo", "nunique"),
    positive_line_gbp=("positive_line_gbp", "sum"),
    negative_line_gbp=("negative_line_gbp", "sum"),
    signed_gbp=("signed_gbp", "sum"),
).reset_index()
display(tables["amount_by_record_type"])

,record_type,customer_id_status,rows,distinct_invoices,positive_line_gbp,negative_line_gbp,signed_gbp
0,regular,identified,397884,18532,8911407.904,0.00,8911407.904
1,regular,missing,132220,1428,1755276.640,0.00,1755276.640
2,exception,identified,8945,3688,0.000,-611342.09,-611342.090
3,exception,missing,2860,2305,0.000,-307594.52,-307594.520


### How do invoice counts and amounts change for the same customer? · `customer_impact`

For each identified customer, calculate two descriptions side by side. One includes every recorded line. The other includes only regular lines with a non-C invoice and positive quantity and price. Count distinct invoice numbers within each description. Starting from all identified customers ensures that those with only exception lines still appear. Missing customer IDs remain in the preceding breakdown and cannot be linked to an individual profile.

### 同一客户在两种口径下，发票数和金额怎样变化？ · `customer_impact`

对每位有编号的客户，分别计算全部记录和常规记录的发票数与金额。常规记录要求发票号不以 C 开头、数量和单价均为正；每种口径均按不同发票号计数。从全部有编号客户出发，可以保留只有异常行的客户。缺少客户编号的行已经在上表单列，无法将它们归入某一位客户的画像。

In [30]:
identified_activity = impact_lines.loc[impact_lines["CustomerID"].notna()]
all_activity_by_id = identified_activity.groupby("CustomerID").agg(
    all_rows=("InvoiceNo", "size"),
    all_invoices=("InvoiceNo", "nunique"),
    last_activity=("InvoiceDate", "max"),
    all_signed_gbp=("signed_gbp", "sum"),
)
regular_activity = identified_activity.loc[identified_activity["record_type"] == "regular"]
regular_activity_by_id = regular_activity.groupby("CustomerID").agg(
    regular_rows=("InvoiceNo", "size"),
    regular_invoices=("InvoiceNo", "nunique"),
    last_regular=("InvoiceDate", "max"),
    regular_gross_gbp=("signed_gbp", "sum"),
)
customer_comparison = all_activity_by_id.join(regular_activity_by_id)
for column in ["regular_rows", "regular_invoices"]:
    customer_comparison[column] = customer_comparison[column].fillna(0).astype("int64")
customer_comparison["regular_gross_gbp"] = customer_comparison["regular_gross_gbp"].fillna(0.0)
customer_comparison["exception_rows"] = customer_comparison["all_rows"] - customer_comparison["regular_rows"]
customer_comparison["amount_difference_gbp"] = customer_comparison["all_signed_gbp"] - customer_comparison["regular_gross_gbp"]

### Recency and the customer-level extract

Measure recency in calendar days from the last observed date under each definition to the day after the final date in the workbook. Retain a missing regular recency when a customer has no regular purchase. The preview includes customer 12346 and one member of each cohort. For 12346, one regular invoice contributes £77,183.60, while the signed arithmetic across two invoices totals £0.00. The exported customer table contains every identified customer. This full-window comparison describes the workbook. Computing predictive features for a model requires a separate cutoff for each query.

### 距最近记录天数与逐客户明细

用工作簿最终记录日期的次日作为共同参照日，分别从每位客户最近一笔常规记录及最近一笔任意记录的日期起计算相隔的日历天数。客户没有常规购买时，相应天数保留为空。下方预览客户 12346 和每个组中的一位客户。12346 的一张常规发票金额为 £77,183.60，两张发票的带符号金额相加为 £0.00。导出的客户表包含所有有编号客户。这是对完整观察期的描述；为模型生成预测特征时，需要使用每个查询各自的历史截止点。

In [31]:
recency_reference = raw["InvoiceDate"].dt.normalize().max() + pd.Timedelta(days=1)
customer_comparison["recency_reference_date"] = recency_reference
for date_column, days_column in [
    ("last_regular", "regular_recency_days"), ("last_activity", "all_recency_days"),
]:
    last_date = customer_comparison.pop(date_column).dt.normalize()
    customer_comparison[days_column] = (recency_reference - last_date).dt.days.astype("Int64")

impact_cohort = pd.Series("mixed", index=customer_comparison.index, dtype="string")
impact_cohort.loc[customer_comparison["exception_rows"] == 0] = "regular_only"
impact_cohort.loc[customer_comparison["regular_rows"] == 0] = "exception_only"
customer_comparison["cohort"] = pd.Categorical(
    impact_cohort, categories=["regular_only", "mixed", "exception_only"],
)
tables["customer_impact"] = customer_comparison.reset_index()
sample_ids = customer_comparison.groupby("cohort", observed=True).head(1).index
display(tables["customer_impact"].loc[tables["customer_impact"]["CustomerID"].isin([*sample_ids, "12346"])])

,CustomerID,all_rows,all_invoices,all_signed_gbp,regular_rows,regular_invoices,regular_gross_gbp,exception_rows,amount_difference_gbp,recency_reference_date,regular_recency_days,all_recency_days,cohort
0,12346,2,2,0.0,1,1,77183.6,1,-77183.6,2011-12-10,326,326,mixed
1,12347,182,7,4310.0,182,7,4310.0,0,0.0,2011-12-10,3,3,regular_only
125,12503,1,1,-1126.0,0,0,0.0,1,-1126.0,2011-12-10,<NA>,338,exception_only


### Which customers are affected? · `customer_impact_summary`

Read the three groups separately. For the 1,563 customers with both regular and exception lines, the median recency is 30 days under regular purchases and 26 days under all activity. The 34 customers with only exceptions have no regular-purchase recency. A group's difference in amounts is its signed worksheet total minus its gross amount on regular lines. No customer segmentation thresholds or model training are used in this comparison.

### 哪些客户受到影响？ · `customer_impact_summary`

分三组查看结果。1,563 位同时有常规行与异常行的客户，按常规购买计算的距最近天数中位数为 30 天，按所有活动计算为 26 天。另有 34 位客户只有异常行，因而没有最近常规购买日期。每组的金额差等于全部行的带符号金额减去常规行金额。本次比较没有设置客户分层阈值，也没有训练模型。

In [32]:
tables["customer_impact_summary"] = customer_comparison.groupby("cohort", observed=False).agg(
    customers=("all_rows", "size"),
    regular_gross_gbp=("regular_gross_gbp", "sum"),
    all_signed_gbp=("all_signed_gbp", "sum"),
    amount_difference_gbp=("amount_difference_gbp", "sum"),
    median_regular_recency_days=("regular_recency_days", "median"),
    median_all_recency_days=("all_recency_days", "median"),
).reset_index()
display(tables["customer_impact_summary"])

,cohort,customers,regular_gross_gbp,all_signed_gbp,amount_difference_gbp,median_regular_recency_days,median_all_recency_days
0,regular_only,2775,2285576.522,2285576.522,0.00,65.0,65.0
1,mixed,1563,6625831.382,6024900.532,-600930.85,30.0,26.0
2,exception_only,34,0.000,-10411.240,-10411.24,<NA>,346.5


**Reading the output**

**`amount_by_record_type`** · Exception lines sum to −£611,342.09 for identified customers and −£307,594.52 where customer ID is missing. These amounts use quantity × unit price and describe ledger entries. Payment and refund settlement are unobserved.

**`customer_impact`** · Comparing each mixed customer's regular lines with all their lines, 1,557 of 1,563 customers gain at least one distinct invoice and 444 have a shorter time since the last record. Both measures are descriptive over the full workbook, with recency measured at 10 December 2011.

**`customer_impact_summary`** · Among 4,372 identified customers, 2,775 have only regular lines, 1,563 have both kinds and 34 have only exceptions. In the mixed group, counting exceptions reduces median days since the last record from 30 to 26.

**如何解读输出**

**`amount_by_record_type`** · 异常行按数量乘单价汇总，有客户编号的部分为 −£611,342.09，缺编号的部分为 −£307,594.52。这是原表记录金额，付款或退款是否到账无法由该表核实。

**`customer_impact`** · 逐人比较仅含常规行与计入全部记录的口径，1,563 位两类都有的客户中，1,557 位的不同发票数增加，444 位距最近记录的天数缩短。这是全期描述性统计，最近性以 2011 年 12 月 10 日为基准。

**`customer_impact_summary`** · 4,372 位已知客户中，2,775 位只有常规行，1,563 位两类都有，34 位只有异常行。对两类都有的客户，计入异常行后，距最近记录的天数中位数从 30 天降至 26 天。

## From original documents to the signed merchandise ledger

The preceding sheets describe the original worksheet, including duplicates, fees and anonymous records. The following sheets implement the project's merchandise policy. Keep the first exact copy across the eight source fields. Remove rows missing required invoice, SKU, date, quantity or price; bad-debt A documents; non-C negative quantities; zero quantities; and nonpositive prices. On this workbook the 1,336 non-C negative-quantity lines have zero price and no customer ID. We exclude them as inventory or adjustment records. Blank notes leave the cause of individual decreases unknown.

Apply the documented literal SKU decisions to sales and credits alike. The code below shows all 18 excluded codes. It keeps `PADS` at £0.001 and excludes `23444` (carriage) and `23574` (packing charge). High quantities remain eligible. The alternative five-leading-digit rule is used only to measure the impact of a code-shape decision.

View A contains signed eligible merchandise rows across the store, with missing IDs preserved and displayed separately as Guest/Unknown. View B uses the same arithmetic for identified customers. Each signed amount uses the row's original quantity and unit price. A C-document number alone cannot identify the customer's earlier sale. [Dataset definition of InvoiceNo](https://archive.ics.uci.edu/dataset/352/online+retail)

## 从原始单据到带符号商品账本

前面的工作表描述原始记录，包含重复、费用及匿名行。以下工作表采用项目商品口径。按八个原始字段删除完全重复行，保留首次出现者。排除缺少发票、商品、日期、数量或价格的行，以及 A 字头坏账单、非 C 负数量行、零数量行和非正单价行。全表 1,336 条非 C 负数量行均为零单价且无客户编号，按库存或调整记录排除；备注为空时，单条记录减少的具体原因仍不明。

销售与贷项共用已冻结的精确商品码规则。下方代码列出 18 个排除编码。保留单价 £0.001 的实物商品 `PADS`，排除 `23444`（配送费）和 `23574`（包装费）。大量购买可正常进入账本。另设“五位数字开头”的对照规则，量化仅凭编码外形判断商品的影响。

视图 A 覆盖全店符合条件的带符号商品行，客户缺失值保留，只在展示时单列为 Guest/Unknown。视图 B 对有客户编号的行采用同样算法。每行金额均按原始数量和原始价格计算。仅凭 C 单的发票号无法确定客户的原购买行。[数据集的 InvoiceNo 定义](https://archive.ics.uci.edu/dataset/352/online+retail)

In [33]:
import numpy as np

EXCLUDED_SKUS = frozenset(['23444', '23574', 'AMAZONFEE', 'B', 'BANK CHARGES', 'C2', 'CRUK', 'D', 'DOT', 'M', 'POST', 'S', 'gift_0001_10', 'gift_0001_20', 'gift_0001_30', 'gift_0001_40', 'gift_0001_50', 'm'])

def _ledger_decisions(raw, stock_policy):
    """Apply mutually exclusive first-match reasons in original worksheet order."""
    if raw["ExcelRow"].isna().any() or not raw["ExcelRow"].is_unique:
        raise ValueError("ExcelRow must be present and unique in the raw worksheet.")
    frame = raw.sort_values("ExcelRow").reset_index(drop=True)
    code = frame["StockCode"].astype("string")
    if stock_policy == "frozen":
        merchandise = ~code.isin(EXCLUDED_SKUS)
    elif stock_policy == "five_digit":
        merchandise = code.str.match(r"^[0-9]{5}", na=False)
    else:
        raise ValueError("stock_policy must be 'frozen' or 'five_digit'.")
    invoice = frame["InvoiceNo"].astype("string")
    credit = invoice.str.startswith("C", na=False)
    quantity = frame["Quantity"]
    price = frame["UnitPrice"]
    checks = [
        ("exact_duplicate", frame.duplicated(RAW_COLUMNS)),
        ("missing_required_field", frame[["InvoiceNo", "StockCode", "InvoiceDate", "Quantity", "UnitPrice"]].isna().any(axis=1)),
        ("bad_debt_document", invoice.str.startswith("A", na=False)),
        ("non_c_negative_quantity", ~credit & quantity.lt(0)),
        ("zero_quantity", quantity.eq(0)),
        ("negative_unit_price", price.lt(0)),
        ("zero_unit_price", price.eq(0)),
        ("excluded_stock_code", ~merchandise),
        ("unexpected_positive_c_line", credit & quantity.gt(0)),
        ("credit", credit & quantity.lt(0)),
    ]
    decisions = np.select(
        [test.to_numpy(dtype=bool, na_value=False) for _, test in checks],
        [reason for reason, _ in checks],
        default="sale",
    )
    return frame, pd.Series(decisions, index=frame.index, name="decision")


def _signed_lines(frame, decisions):
    eligible = decisions.isin(["sale", "credit"])
    ledger = frame.loc[eligible].copy()
    ledger["ledger_type"] = decisions.loc[eligible]
    # Guest is a display value only. CustomerID stays missing for all joins.
    ledger["customer_display"] = ledger["CustomerID"].astype("string").fillna("Guest/Unknown")
    ledger["signed_amount_gbp"] = ledger["Quantity"] * ledger["UnitPrice"]
    return ledger.reset_index(drop=True)


def build_signed_ledger(raw: pd.DataFrame, stock_policy="frozen") -> pd.DataFrame:
    """Build store-level signed view A. View B selects its identified customers."""
    frame, decisions = _ledger_decisions(raw, stock_policy)
    return _signed_lines(frame, decisions)

## Select a purchase row for each merchandise credit

Process credits in timestamp and Excel-row order. Among sales with the same known customer and exact SKU, choose the most recent eligible sale with enough remaining units to cover the entire credit. No splitting across sales. At a tied purchase time, take the smallest eligible worksheet row, preferring a row before the credit if one exists. Same-minute matches from later worksheet rows are allowed and explicitly flagged. Unit price is retained as evidence and may differ.

The credit's destination determines the maximum elapsed time for the **cancellation proxy**: UK 72 hours, listed European destinations 120 hours, other known destinations 240 hours. Use continuous elapsed time, inclusive of each threshold. A later match is the **return proxy**. Missing/unspecified destinations, known customers without a match and anonymous credits remain distinguishable. Every matched credit consumes reversible capacity and reduces the sale’s retained quantity when the credit becomes visible. This includes classified returns and matches to unknown destinations. UCI identifies the C prefix as cancellation; the workbook has no dispatch, delivery or physical-return timestamp to validate these subtypes.

## 为每条商品贷项选择可对应的购买行

按贷项时间和 Excel 行号顺序处理。同一已知客户、同一精确 SKU 下，在时间不晚于贷项且可冲数量足以覆盖整条贷项的销售行中，选择最近的一条。不得分摊到多条销售行。购买时间相同时，优先选贷项行之前的行，再按符合条件者中最小的 Excel 行号决定。同分钟内排在贷项之后的销售行也可匹配，另设标记。单价保留为核查依据，可以不相等。

根据贷项的目的地确定**操作性取消**时限：英国 72 小时，所列欧洲目的地 120 小时，其他已知目的地 240 小时。用实际经过的连续时间比较，等于时限也计入取消；超过时限的匹配标为**操作性退货**。地区缺失或未指定、已知客户未匹配、匿名贷项分别保留。每笔已匹配的贷项从出现时起扣减可冲余额及对应购买行的保留数量，包括操作性退货和地区未知的已匹配贷项。UCI 说明 C 前缀表示取消，工作簿缺少发货、签收和实物退回时间，无法据此验证这些细分标签。

In [34]:
EUROPE_COUNTRIES = frozenset({
    "EIRE", "France", "Germany", "Netherlands", "Belgium", "Spain", "Switzerland",
    "Portugal", "Channel Islands", "Italy", "Norway", "Sweden", "Denmark",
    "Finland", "Austria", "Poland", "Cyprus", "Greece", "Czech Republic",
    "Malta", "Iceland", "European Community", "Lithuania",
})


SCENARIOS = {
    "five_digit_independent_strict": ("five_digit", False, False),
    "five_digit_independent_inclusive": ("five_digit", False, True),
    "frozen_independent_inclusive": ("frozen", False, True),
    "frozen_balance_strict": ("frozen", True, False),
    "frozen_balance_inclusive": ("frozen", True, True),
}


CREDIT_STATUSES = ("cancellation", "return", "unmatched", "anonymous", "unknown_region")


def _region(country, uk_hours=72):
    """Map the C document's destination to its continuous elapsed-time threshold."""
    if pd.isna(country) or country in ("", "Unspecified"):
        return "unknown", None
    if country == "United Kingdom":
        return "UK", uk_hours
    if country in EUROPE_COUNTRIES:
        return "Europe", 120
    return "ROW", 240


def match_reversals(
    ledger: pd.DataFrame, *, use_balance=True, include_same_time=True, uk_hours=72,
) -> pd.DataFrame:
    """Match each C once to the nearest eligible single positive line.

    Work in credit timestamp and worksheet-row order. A sale must have the
    same original customer and SKU, with enough reversible units on that
    one row. Price is evidence but is not a matching condition. At the nearest
    eligible purchase time, ascending ExcelRow prefers rows before the C and
    then the smallest row. Inclusive matching allows an equal-time row appearing
    later on the sheet. Every match reserves units on its source line. The
    regional time threshold labels the match for operational reporting. A
    net-retained snapshot deducts linked units once the credit is observable,
    including returns and matches with an unknown destination. For sensitivity
    checks, uk_hours changes the UK reporting threshold alone (whole hours).
    """
    if not isinstance(uk_hours, (int, np.integer)) or uk_hours < 0:
        raise ValueError("uk_hours must be a nonnegative whole number of hours.")
    credits = ledger.loc[ledger["ledger_type"].eq("credit")].sort_values(["InvoiceDate", "ExcelRow"])
    keys = ["CustomerID", "StockCode"]
    relevant = credits[keys].dropna().drop_duplicates()
    sales = ledger.loc[ledger["ledger_type"].eq("sale")].merge(relevant, on=keys, validate="many_to_one")
    # Newest eligible time first; the lower original row wins equal-time ties.
    sales = sales.sort_values(["InvoiceDate", "ExcelRow"], ascending=[False, True])
    groups = {}
    for sale in sales.itertuples(index=False):
        groups.setdefault((sale.CustomerID, sale.StockCode), []).append(sale)
    remaining = {}
    records = []
    for credit in credits.itertuples(index=False):
        customer, sku = credit.CustomerID, credit.StockCode
        units = -credit.Quantity
        region, hours = _region(credit.Country, uk_hours=uk_hours)
        record = {
            "credit_excel_row": credit.ExcelRow, "credit_invoice_no": credit.InvoiceNo,
            "credit_time": credit.InvoiceDate, "CustomerID": customer, "StockCode": sku,
            "Country": credit.Country, "region": region, "threshold_hours": hours,
            "credited_units": units, "credit_unit_price": credit.UnitPrice,
            "credit_amount_gbp": credit.signed_amount_gbp,
            "purchase_excel_row": None, "purchase_invoice_no": None, "purchase_time": None,
            "purchase_unit_price": None, "purchase_quantity": None, "balance_before": None,
            "balance_after": None, "lag_minutes": None, "lag_days": None,
            "same_time": False, "later_excel_row_same_time": False,
            "status": "anonymous" if pd.isna(customer) else "unmatched",
        }
        if pd.notna(customer):
            for sale in groups.get((customer, sku), ()):
                lag = credit.InvoiceDate - sale.InvoiceDate
                if lag < pd.Timedelta(0) or (lag == pd.Timedelta(0) and not include_same_time):
                    continue
                available = remaining.get(sale.ExcelRow, sale.Quantity)
                if available < units:
                    continue
                after = available - units
                if use_balance:
                    remaining[sale.ExcelRow] = after
                status = "unknown_region"
                if hours is not None:
                    status = "cancellation" if lag <= pd.Timedelta(hours=hours) else "return"
                record.update({
                    "purchase_excel_row": sale.ExcelRow, "purchase_invoice_no": sale.InvoiceNo,
                    "purchase_time": sale.InvoiceDate, "purchase_unit_price": sale.UnitPrice,
                    "purchase_quantity": sale.Quantity, "balance_before": available,
                    "balance_after": after, "lag_minutes": lag.total_seconds() / 60,
                    "lag_days": lag / pd.Timedelta(days=1), "same_time": lag == pd.Timedelta(0),
                    "later_excel_row_same_time": lag == pd.Timedelta(0) and sale.ExcelRow > credit.ExcelRow,
                    "status": status,
                })
                break
        records.append(record)
    columns = [
        "credit_excel_row", "credit_invoice_no", "credit_time", "CustomerID", "StockCode",
        "Country", "region", "threshold_hours", "credited_units", "credit_unit_price",
        "credit_amount_gbp", "purchase_excel_row", "purchase_invoice_no", "purchase_time",
        "purchase_unit_price", "purchase_quantity", "balance_before", "balance_after",
        "lag_minutes", "lag_days", "same_time", "later_excel_row_same_time", "status",
    ]
    result = pd.DataFrame.from_records(records, columns=columns)
    for name in ("credit_time", "purchase_time"):
        result[name] = pd.to_datetime(result[name])
    for name in ("credit_excel_row", "threshold_hours", "credited_units", "purchase_excel_row",
                 "purchase_quantity", "balance_before", "balance_after"):
        result[name] = result[name].astype("Int64")
    for name in ("credit_amount_gbp", "credit_unit_price", "purchase_unit_price", "lag_minutes", "lag_days"):
        result[name] = result[name].astype(float)
    for name in ("same_time", "later_excel_row_same_time"):
        result[name] = result[name].astype(bool)
    return result

## Reconcile scenarios and trace individual differences

Calculate row counts, signed units and GBP amounts separately for each scenario. Compare the supplied five-digit, independent matching benchmark with the frozen SKU register and remaining-quantity matching. Compare strict timestamps against inclusive same-minute matching with other settings fixed. A negative signed credit retains its recorded price in the ledger. For a cancellation, gross units removed from the original sale are valued at that sale's price. Keep those two figures separate throughout.

The next functions construct the controls. `reported_*` values are comparison targets from the project discussion. `observed_*` values are computed from the workbook. The discussion's regional “unmatched” total combines identified unmatched and anonymous rows, so only the regional comparison combines those two statuses.

## 复核对照情景并追查单行差异

分别计算每个情景的行数、带符号数量及英镑金额。把讨论稿中的“五位数字、独立匹配”基准与冻结商品码、可冲余额匹配进行对照。在其他设置相同时比较严格早于与允许同分钟的差别。账本保留贷项按其原始单价计算的负金额；核算取消时，被扣减的原销售数量按原销售单价折算。两者在表中分别核算。

以下函数生成对照结果。`reported_*` 列是讨论稿提供的基准，`observed_*` 列根据本工作簿计算。讨论稿的地区“未匹配”合并了已知客户未匹配与匿名记录，只有地区基准对照按此口径合并。

In [35]:
def _scenario_summary(scenario, ledger, matches, use_balance):
    """Reconcile signed documents and sale-price valuations over the full period.

    Gross after cancellations measures sales before returns. Net-retained units
    deduct every allocated credit from its source sale. Signed view A/B instead
    sums each document at its own recorded price, including unlinked credits.
    """
    rows = []

    def record(metric, frame, units="Quantity", money="signed_amount_gbp"):
        rows.append({
            "scenario": scenario, "metric": metric, "rows": len(frame),
            "units": frame[units].sum(), "signed_amount_gbp": frame[money].sum(),
        })

    known = ledger.loc[ledger["CustomerID"].notna()]
    sales = ledger.loc[ledger["ledger_type"].eq("sale")]
    record("view_a", ledger)
    record("view_b", known)
    record("guest", ledger.loc[ledger["CustomerID"].isna()])
    record("sale", sales)
    record("credit", ledger.loc[ledger["ledger_type"].eq("credit")])
    record("known_observed_sales", sales.loc[sales["CustomerID"].notna()])
    statuses = matches.assign(signed_units=-matches["credited_units"])
    for status in CREDIT_STATUSES:
        selected = statuses.loc[statuses["status"].eq(status)]
        record(status, selected, units="signed_units", money="credit_amount_gbp")
    if use_balance:
        linked = matches.loc[matches["purchase_excel_row"].notna()]
        deductions = {
            "gross_after_cancellations": linked.loc[linked["status"].eq("cancellation")],
            "net_retained_at_sale_prices": linked,
        }
        for metric, credits in deductions.items():
            allocated = credits.groupby("purchase_excel_row")["credited_units"].sum()
            quantity = sales["Quantity"] - sales["ExcelRow"].map(allocated).fillna(0)
            goods = sales.loc[quantity.gt(0)].assign(effective_units=quantity)
            goods["effective_amount_gbp"] = goods["effective_units"] * goods["UnitPrice"]
            record(metric, goods, "effective_units", "effective_amount_gbp")
            record("known_" + metric, goods.loc[goods["CustomerID"].notna()],
                   "effective_units", "effective_amount_gbp")
    return pd.DataFrame(rows)


def _compare_matches(before_name, before, after_name, after):
    """Trace lines whose inclusion, selected source, classification or balance changed."""
    detail = ["status", "purchase_excel_row", "purchase_invoice_no", "purchase_time",
              "purchase_unit_price", "lag_minutes", "balance_before", "balance_after",
              "same_time", "later_excel_row_same_time"]
    ids = ["credit_excel_row", "credit_invoice_no", "credit_time", "CustomerID", "StockCode", "region",
           "credited_units", "credit_amount_gbp"]
    identity = pd.concat([before[ids], after[ids]], ignore_index=True).drop_duplicates("credit_excel_row")
    key = "credit_excel_row"
    side = before.set_index(key)[detail].add_prefix("before_").join(
        after.set_index(key)[detail].add_prefix("after_"), how="outer",
    )
    changes = {}
    for field in ["status", "purchase_excel_row", "balance_before", "balance_after"]:
        old, new = side["before_" + field], side["after_" + field]
        changes[field] = (old.isna() != new.isna()) | old.ne(new).fillna(False)
    side["status_changed"] = changes["status"]
    side["purchase_changed"] = changes["purchase_excel_row"]
    side["balance_changed"] = changes["balance_before"] | changes["balance_after"]
    relevant = side["status_changed"] | side["purchase_changed"] | side["balance_changed"]
    result = identity.merge(side.loc[relevant], on=key, how="inner")
    result.insert(0, "after_scenario", after_name)
    result.insert(0, "before_scenario", before_name)
    return result.sort_values(key, ignore_index=True)

In [36]:
def _reference_benchmark(summary, regional):
    """Compare supplied figures with measured outputs without fitting the policy to them."""
    targets = {
        "view_a": (531172, 9770919.71), "view_b": (399656, 8265476.93),
        "credit": (8668, None), "cancellation": (1481, -312072.49),
        "return": (5913, -134674.78), "unmatched": (1112, -25003.44),
        "anonymous": (162, -4150.45),
    }
    rows = []

    def add(scenario, scope, metric, expected_rows, expected_amount, observed_rows, observed_amount):
        rows.append({
            "scenario": scenario, "scope": scope, "metric": metric, "reported_rows": expected_rows,
            "observed_rows": observed_rows, "delta_rows": observed_rows - expected_rows,
            "reported_amount_gbp": expected_amount, "observed_amount_gbp": observed_amount,
            "delta_amount_gbp": None if expected_amount is None else observed_amount - expected_amount,
        })

    for row in summary.itertuples(index=False):
        if row.metric in targets:
            count, amount = targets[row.metric]
            add(row.scenario, "all", row.metric, count, amount, row.rows, row.signed_amount_gbp)
    # The supplied regional 'unmatched' totals include anonymous credits. The
    # main ledger retains the separate unmatched, anonymous and unknown-region states.
    region_targets = {"UK": (1111, 5084, 1134), "Europe": (194, 788, 133), "ROW": (176, 41, 7)}
    for scenario in SCENARIOS:
        for region, counts in region_targets.items():
            group = regional.loc[regional["scenario"].eq(scenario) & regional["region"].eq(region)]
            for metric, statuses, reported in zip(
                ["cancellation", "return", "unmatched_including_anonymous"],
                [["cancellation"], ["return"], ["unmatched", "anonymous"]], counts,
            ):
                selected = group.loc[group["status"].isin(statuses)]
                add(scenario, region, metric, reported, None,
                    selected["rows"].sum(), selected["signed_amount_gbp"].sum())
    return pd.DataFrame(rows)

In [37]:
def compute_reversal_tables(raw: pd.DataFrame) -> dict[str, pd.DataFrame]:
    """Classify the original eight-field worksheet with its stable ExcelRow locators.

    Return canonical signed A/B, canonical credit allocations, a source audit and
    the benchmark controls. Raw worksheet counts, independently classified credits
    and balance-reserved credits have different denominators. Every comparison
    below preserves its own source policy, lag rule and allocation convention.
    """
    sources, decisions, audits = {}, {}, []
    for policy in ("five_digit", "frozen"):
        frame, decision = _ledger_decisions(raw, policy)
        decisions[policy] = decision
        sources[policy] = _signed_lines(frame, decision)
        audit = pd.DataFrame({
            "reason": decision, "customer_id_status": np.where(frame["CustomerID"].isna(), "anonymous", "known"),
            "signed_amount_gbp": frame["Quantity"] * frame["UnitPrice"],
        })
        audit = audit.groupby(["reason", "customer_id_status"], as_index=False).agg(
            rows=("reason", "size"), signed_amount_gbp=("signed_amount_gbp", "sum"),
        )
        audit.insert(0, "stock_policy", policy)
        audits.append(audit)

    difference = decisions["five_digit"].ne(decisions["frozen"])
    detail = frame.loc[difference].copy()
    detail["five_digit_decision"] = decisions["five_digit"].loc[difference]
    detail["frozen_decision"] = decisions["frozen"].loc[difference]
    detail["signed_amount_gbp"] = detail["Quantity"] * detail["UnitPrice"]

    matches, summaries, regions = {}, [], []
    for name, (policy, balance, same_time) in SCENARIOS.items():
        match = match_reversals(sources[policy], use_balance=balance, include_same_time=same_time)
        matches[name] = match
        summaries.append(_scenario_summary(name, sources[policy], match, balance))
        region = match.groupby(["region", "status"], as_index=False).agg(
            rows=("status", "size"), signed_amount_gbp=("credit_amount_gbp", "sum"),
        )
        region.insert(0, "scenario", name)
        regions.append(region)
    comparisons = [
        ("five_digit_independent_strict", "five_digit_independent_inclusive"),
        ("five_digit_independent_inclusive", "frozen_independent_inclusive"),
        ("frozen_independent_inclusive", "frozen_balance_inclusive"),
        ("frozen_balance_strict", "frozen_balance_inclusive"),
    ]
    changes = [
        _compare_matches(before, matches[before], after, matches[after]) for before, after in comparisons
    ]
    summary = pd.concat(summaries, ignore_index=True)
    region_summary = pd.concat(regions, ignore_index=True)
    signed = sources["frozen"]
    tables = {
        "signed_ledger_a": signed,
        "signed_ledger_b": signed.loc[signed["CustomerID"].notna()].reset_index(drop=True),
        "credit_allocations": matches["frozen_balance_inclusive"],
        "source_audit": pd.concat(audits, ignore_index=True),
        "source_differences": detail.reset_index(drop=True),
        "reconciliation": summary,
        "region_summary": region_summary,
        "match_differences": pd.concat(changes, ignore_index=True),
        "benchmark": _reference_benchmark(summary, region_summary),
    }
    # Keep sub-penny source prices. Remove floating-point summation noise from
    # published diagnostics without modifying source prices or the signed ledgers.
    for name in ["source_audit", "reconciliation", "region_summary", "benchmark"]:
        money = [column for column in tables[name] if column.endswith("_gbp")]
        rounded = tables[name][money].round(6)
        # A rounded zero has a single CSV representation across NumPy versions.
        tables[name][money] = rounded.mask(rounded.eq(0), 0.0)
    return tables

### From matched credits to retained purchases

Start with identified merchandise sales. For each original sale, sum all linked credit units already visible at the exclusive cutoff. Subtract once. A zero balance removes the line. A partial balance keeps the remaining units, valued at the original sale price. Complete the invoice using the latest eligible original sale time before applying credits, then combine retained lines by customer, invoice and SKU. A different line in the same basket can still carry the item. The calculation below also keeps cancellation-only gross quantities for the accounting comparison.

### 从贷项匹配到净保留购买

从已知客户的商品销售行出发，对每条原始销售汇总截止点之前已经出现的所有匹配贷项数量，只扣一次。余额为零就移出该行；部分冲销后，按原销售单价计算剩余数量的金额。先用原始合格购买行的最晚时间确定整篮完成时点，再将保留行按客户、发票、商品汇总。即使某行归零，同篮另一行仍可能保留该商品。下方也单独计算仅扣取消的毛数量，用于账务对照。

In [38]:
INVOICE_KEYS = ["CustomerID", "InvoiceNo"]


def purchase_line_balances(ledger, allocations, *, cutoff=None):
    """Return all identified positive source lines in completed baskets, including fully credited ones.

    Both inputs must use canonical allocation results and Excel worksheet row
    identifiers. For historical replay, pass a cutoff. Only baskets completed
    and credits observed strictly before that timestamp enter.
    None requests a full-workbook retrospective view. The immutable invoice
    completion is computed before subtracting any matched credits. Allocation to
    the source line determines the deduction. Elapsed-time classes describe it.
    """
    sale = ledger["ledger_type"].eq("sale") & ledger["CustomerID"].notna()
    lines = ledger.loc[sale, ["ExcelRow", *RAW_COLUMNS]].copy()
    lines["InvoiceDate"] = pd.to_datetime(lines["InvoiceDate"], format="mixed")
    lines["basket_time"] = lines.groupby(INVOICE_KEYS, sort=False)["InvoiceDate"].transform("max")
    effective = allocations.loc[allocations["purchase_excel_row"].notna()]
    if cutoff is not None:
        end = pd.Timestamp(cutoff)
        lines = lines.loc[lines["basket_time"].lt(end)].copy()
        credit_dates = pd.to_datetime(effective["credit_time"], format="mixed")
        effective = effective.loc[credit_dates.lt(end)]

    lines["original_quantity"] = lines["Quantity"]
    for status, column in [("cancellation", "cancelled_units"), ("return", "returned_units")]:
        totals = (
            effective.loc[effective["status"].eq(status)]
            .groupby("purchase_excel_row")["credited_units"].sum()
        )
        lines[column] = lines["ExcelRow"].map(totals).fillna(0).astype("int64")
    matched = effective.groupby("purchase_excel_row")["credited_units"].sum()
    lines["matched_credit_units"] = lines["ExcelRow"].map(matched).fillna(0).astype("int64")
    lines["other_matched_units"] = lines["matched_credit_units"] - lines["cancelled_units"] - lines["returned_units"]
    lines["Quantity"] = lines["original_quantity"] - lines["matched_credit_units"]
    if lines["Quantity"].lt(0).any():
        raise ValueError("Matched credit units exceed the original quantity on a sale line.")
    components = {"cancelled_units": "cancellation", "returned_units": "return", "other_matched_units": "other"}
    lines["matched_credit_composition"] = "none"
    for col, label in components.items():
        lines.loc[lines[col].gt(0), "matched_credit_composition"] = label
    mixed = lines[list(components)].gt(0).sum(axis=1).gt(1)
    lines.loc[mixed, "matched_credit_composition"] = "mixed"
    lines["line_amount_gbp"] = lines["Quantity"] * lines["UnitPrice"]
    return lines.sort_values("ExcelRow", ignore_index=True)

In [39]:
def retention_comparison(balances):
    """Compare the three purchase definitions on the same identified, eligible sale lines."""
    rows = []
    event_keys = ["CustomerID", "InvoiceNo", "StockCode"]
    definitions = {
        "observed_positive_sales": balances["original_quantity"],
        "gross_after_cancellations": balances["original_quantity"] - balances["cancelled_units"],
        "net_after_all_matched_credits": balances["Quantity"],
    }
    for name, qty in definitions.items():
        kept = balances.loc[qty.gt(0)]
        rows.append({
            "definition": name, "sale_lines": len(kept), "basket_sku_events": len(kept[event_keys].drop_duplicates()),
            "baskets": len(kept[["CustomerID", "InvoiceNo"]].drop_duplicates()), "customers": kept["CustomerID"].nunique(),
            "units": int(qty.sum()), "value_at_sale_prices_gbp": round((qty * balances["UnitPrice"]).sum(), 6),
        })
    return pd.DataFrame(rows)


def retention_transitions(balances):
    """Count fully, partially and unaffected source lines and basket–SKU groups."""
    qty_cols = ["original_quantity", "cancelled_units", "returned_units", "other_matched_units", "Quantity"]
    events = balances.groupby(["CustomerID", "InvoiceNo", "StockCode"], sort=False)[qty_cols].sum()
    summaries = []
    for unit, data in [("source_sale_line", balances), ("basket_sku_event", events)]:
        components = data[["cancelled_units", "returned_units", "other_matched_units"]].gt(0)
        state = pd.Series("unaffected", index=data.index)
        state.loc[data["Quantity"].lt(data["original_quantity"])] = "partial"
        state.loc[data["Quantity"].eq(0)] = "full"
        composition = pd.Series("none", index=data.index)
        for column, label in [("cancelled_units", "cancellation"), ("returned_units", "return"), ("other_matched_units", "other")]:
            composition.loc[components[column]] = label
        composition.loc[components.sum(axis=1).gt(1)] = "mixed"
        summary = data.assign(state=state, credit_composition=composition).groupby(
            ["state", "credit_composition"], as_index=False
        ).agg(count=("Quantity", "size"), observed_units=("original_quantity", "sum"), retained_units=("Quantity", "sum"))
        summary.insert(0, "unit", unit)
        summary["all_observed"] = len(data)
        summary["share_of_all_observed"] = summary["count"] / len(data)
        summaries.append(summary)
    return pd.concat(summaries, ignore_index=True)

### Count returns and later purchases at two levels

A return credit has a recorded quantity. The corresponding sale row can receive several credits. Count individual credits relative to the original sale quantity, then separately group credits by sale and evaluate the final balance. For each follow-up, a later sale must have the same identified customer and SKU, a different purchase invoice and a strictly later line timestamp than the selected credit. The observed outcome includes any later eligible sale. The net outcome requires at least one such line to retain units through the end of the workbook.

### 按两种粒度统计退货及之后的购买

一条退货贷项有自己的记录数量，一条原销售行却可接收多笔贷项。先比较单笔贷项数量与原销售数量，再按原销售行汇总贷项并判断最终余额。追查后续购买时，要求客户已知且相同、SKU 相同、购买发票不同，且购买行时间严格晚于所选贷项。观察口径统计其后任何合格销售。净保留口径还要求至少一条后续销售到工作簿期末仍有正余额。

In [40]:
def compare_return_credit_rows(scenarios):
    """Reproduce the discussion's two statistics with the same per-credit definition."""
    rows = []
    for scenario, matches, ledger in scenarios:
        returns = matches.loc[matches["status"].eq("return")].copy()
        returns["size_relative_to_original"] = returns["credited_units"].eq(returns["purchase_quantity"]).map(
            {True: "full", False: "partial"}
        )
        # A sale on a different invoice is a follow-up if any of its lines is strictly later.
        sales = ledger.loc[ledger["ledger_type"].eq("sale") & ledger["CustomerID"].notna()]
        orders = sales.groupby(["CustomerID", "StockCode", "InvoiceNo"], as_index=False)["InvoiceDate"].max()
        pairs = returns.merge(orders, on=["CustomerID", "StockCode"])
        later = pairs["InvoiceNo"].ne(pairs["purchase_invoice_no"]) & pairs["InvoiceDate"].gt(pairs["credit_time"])
        followed = pairs.loc[later, "credit_excel_row"]
        returns["later_observed"] = returns["credit_excel_row"].isin(followed)
        result = returns.groupby("size_relative_to_original", as_index=False).agg(
            return_credit_rows=("credit_excel_row", "size"), distinct_linked_sale_lines=("purchase_excel_row", "nunique"),
            later_observed=("later_observed", "sum")
        )
        result["observed_rate"] = result["later_observed"] / result["return_credit_rows"]
        result["scenario"] = scenario
        result["scope"] = "individual_return_credit_rows"
        result["metric"] = "return_credit_relative_to_original_sale_quantity"
        rows.append(result)
    return pd.concat(rows, ignore_index=True)


def return_repurchase(balances, allocations):
    """Compare follow-up definitions at the credit-event and distinct original-sale level."""
    returns = allocations.loc[allocations["status"].eq("return")].copy()
    balances_by_row = balances.set_index("ExcelRow")
    matched = allocations.dropna(subset=["purchase_excel_row"]).sort_values(["credit_time", "credit_excel_row"])
    anchors = []
    for label, credits, keep in [
        ("first_operational_return", returns, "first"),
        ("last_operational_return", returns, "last"),
        ("each_operational_return", returns, None),
        ("last_matched_credit", matched, "last"),
    ]:
        ordered = credits.sort_values(["credit_time", "credit_excel_row"])
        if keep is not None:
            ordered = ordered.drop_duplicates("purchase_excel_row", keep=keep)
        anchored = ordered[["credit_excel_row", "credit_time", "purchase_excel_row",
                            "purchase_invoice_no", "CustomerID", "StockCode"]].copy()
        anchored["anchor"] = label
        anchored["final_quantity"] = anchored["purchase_excel_row"].map(balances_by_row["Quantity"])
        anchors.append(anchored)
    all_anchors = pd.concat(anchors, ignore_index=True).rename_axis("anchor_id").reset_index()
    # One candidate per invoice. The latest line time determines whether any line is strictly later.
    order_keys = ["CustomerID", "StockCode", "InvoiceNo"]
    sales = balances.groupby(order_keys, as_index=False)["InvoiceDate"].max().rename(columns={"InvoiceDate": "observed_time"})
    retained = balances.loc[balances["Quantity"].gt(0)].groupby(order_keys)["InvoiceDate"].max().rename("net_time")
    orders = sales.join(retained, on=order_keys)
    pairs = all_anchors.merge(orders, on=["CustomerID", "StockCode"], how="left")
    distinct = pairs["InvoiceNo"].ne(pairs["purchase_invoice_no"])
    pairs["later_observed"] = distinct & pairs["observed_time"].gt(pairs["credit_time"])
    pairs["later_net"] = distinct & pairs["net_time"].gt(pairs["credit_time"])
    outcomes = pairs.groupby("anchor_id")[["later_observed", "later_net"]].any()
    annotated = all_anchors.join(outcomes, on="anchor_id")
    annotated["source_outcome"] = annotated["final_quantity"].gt(0).map({True: "partial", False: "full"})
    summary = annotated.groupby(["anchor", "source_outcome"], as_index=False).agg(
        denominator=("anchor_id", "size"), distinct_source_sales=("purchase_excel_row", "nunique"),
        later_observed=("later_observed", "sum"), later_net=("later_net", "sum"),
    )
    summary["unit"] = summary["anchor"].eq("each_operational_return").map({True: "return_credit_row", False: "source_sale_line"})
    summary["observed_rate"] = summary["later_observed"] / summary["denominator"]
    summary["net_rate"] = summary["later_net"] / summary["denominator"]
    return summary

In [41]:
def reversal_examples(allocations):
    """Trace each matched credit on selected source lines in the order it became visible."""
    matches = allocations.dropna(subset=["purchase_excel_row"]).sort_values(["credit_time", "credit_excel_row"]).copy()
    cancelled = matches["status"].eq("cancellation")
    matches["cancelled_units_here"] = matches["credited_units"].where(cancelled, 0)
    matches["gross_deduction_gbp_here"] = matches["cancelled_units_here"] * matches["purchase_unit_price"]
    matches["net_deduction_gbp_here"] = matches["credited_units"] * matches["purchase_unit_price"]
    cumulative = matches.groupby("purchase_excel_row")["cancelled_units_here"].cumsum()
    matches["gross_units_after_cancellations"] = matches["purchase_quantity"] - cumulative
    matches["purchase_units_after_credit"] = matches["balance_after"]
    cutoff = pd.Timestamp("2011-10-01")
    visible = matches.loc[matches["credit_time"].lt(cutoff)].groupby("purchase_excel_row")["credited_units"].sum()
    matches["purchase_units_at_oct1"] = (
        matches["purchase_quantity"] - matches["purchase_excel_row"].map(visible).fillna(0)
    ).where(matches["purchase_time"].lt(cutoff)).astype("Int64")

    selected = matches.loc[
        matches["credit_invoice_no"].isin(["C541433", "C581484"])
        | matches["credit_excel_row"].eq(222694), "purchase_excel_row"
    ]
    partial = matches.loc[cancelled & matches["credited_units"].lt(matches["balance_before"]), "purchase_excel_row"].head(1)
    example_rows = [67, 292, 5249, 34475, 321282, 370990, 371018, 373424]
    return matches.loc[matches["purchase_excel_row"].isin([*selected, *partial, *example_rows])].reset_index(drop=True)


def reversal_worksheets(raw):
    evidence = compute_reversal_tables(raw)
    main = "frozen_balance_inclusive"
    summary = evidence["reconciliation"].loc[lambda x: x["scenario"].eq(main)].drop(columns="scenario")
    summary = summary.assign(section="ledger_and_matching")
    source = evidence["source_audit"].loc[lambda x: x["stock_policy"].eq("frozen")].drop(columns="stock_policy")
    source = source.rename(columns={"reason": "metric"}).assign(section="raw_first_rule")
    overview = pd.concat([summary, source], ignore_index=True)
    allocations = evidence["credit_allocations"]
    historical_ledger = build_signed_ledger(raw, stock_policy="five_digit")
    historical = match_reversals(historical_ledger, use_balance=False, include_same_time=True)
    per_credit = compare_return_credit_rows([
        ("five_digit_independent_inclusive", historical, historical_ledger),
        (main, allocations, evidence["signed_ledger_b"]),
    ])
    controls = pd.concat([evidence["benchmark"], per_credit], ignore_index=True)
    comparisons, transitions = [], []
    for window, cutoff in [("training_before_october", "2011-10-01"), ("whole_workbook", None)]:
        balances = purchase_line_balances(evidence["signed_ledger_b"], allocations, cutoff=cutoff)
        comparisons.append(retention_comparison(balances).assign(window=window))
        transitions.append(retention_transitions(balances).assign(window=window))
        if cutoff is None:
            repurchases = return_repurchase(balances, allocations)
    return {
        "ledger_summary": overview[["section", "metric", "customer_id_status", "rows", "units", "signed_amount_gbp"]],
        "credit_summary": evidence["region_summary"].loc[lambda x: x["scenario"].eq(main)].drop(columns="scenario"),
        "credit_allocations": allocations,
        "credit_controls": controls,
        "credit_control_rows": evidence["match_differences"],
        "reversal_examples": reversal_examples(allocations),
        "sku_policy_differences": evidence["source_differences"],
        "retention_comparison": pd.concat(comparisons, ignore_index=True),
        "retention_transitions": pd.concat(transitions, ignore_index=True),
        "return_repurchase": repurchases,
    }


tables.update(reversal_worksheets(raw))

### Which value belongs to which calculation?

Across the workbook, signed view A contains 531,112 merchandise rows and £9,770,043.163 including anonymous customers. Identified view B has 399,563 rows and £8,264,201.933. Each credit uses its own recorded price. For identified customers, cancellation-only gross is £8,386,326.743 at original sale prices. Net retained purchases, after all linked credits, are 4,893,993 units valued at £8,252,857.003. The original worksheet contains sub-penny prices. The source audit underneath the totals shows each line’s first exclusion.

### 各个金额分别来自什么计算？

全期带符号视图 A 包含 531,112 行商品交易，含匿名客户的金额为 £9,770,043.163。已知客户视图 B 含 399,563 行，金额为 £8,264,201.933。每笔贷项采用其记录单价。对已知客户，仅扣取消、按原售价计算的毛额为 £8,386,326.743。扣掉全部已匹配贷项后，净保留 4,893,993 件，按原售价计 £8,252,857.003。原始工作簿含不足一便士的单价。汇总下方的来源审计列出各行最先命中的排除规则。

In [42]:
ledger_summary = tables["ledger_summary"]
display(ledger_summary[ledger_summary["section"].eq("ledger_and_matching")])
display(ledger_summary[ledger_summary["section"].eq("raw_first_rule")])

,section,metric,customer_id_status,rows,units,signed_amount_gbp
0,ledger_and_matching,view_a,NaN,531112,5291280.0,9.770043e+06
1,ledger_and_matching,view_b,NaN,399563,4873986.0,8.264202e+06
2,ledger_and_matching,guest,NaN,131549,417294.0,1.505841e+06
3,ledger_and_matching,sale,NaN,522447,5561294.0,1.024591e+07
4,ledger_and_matching,credit,NaN,8665,-270014.0,-4.758712e+05
5,ledger_and_matching,known_observed_sales,NaN,391060,5141694.0,8.735923e+06
6,ledger_and_matching,cancellation,NaN,1473,-189315.0,-3.118121e+05
7,ledger_and_matching,return,NaN,5883,-58386.0,-1.337952e+05
8,ledger_and_matching,unmatched,NaN,1147,-20007.0,-2.611339e+04
9,ledger_and_matching,anonymous,NaN,162,-2306.0,-4.150450e+03


,section,metric,customer_id_status,rows,units,signed_amount_gbp
15,raw_first_rule,bad_debt_document,anonymous,3,NaN,-11062.060
16,raw_first_rule,credit,anonymous,162,NaN,-4150.450
17,raw_first_rule,credit,known,8503,NaN,-471720.710
18,raw_first_rule,exact_duplicate,anonymous,43,NaN,194.590
19,raw_first_rule,exact_duplicate,known,5225,NaN,21546.390
20,raw_first_rule,excluded_stock_code,anonymous,1015,NaN,-47291.640
21,raw_first_rule,excluded_stock_code,known,2001,NaN,14317.491
22,raw_first_rule,non_c_negative_quantity,anonymous,1336,NaN,0.000
23,raw_first_rule,sale,anonymous,131387,NaN,1509991.680
24,raw_first_rule,sale,known,391060,NaN,8735922.643


### Separate evidence from the regional interpretation

The 8,665 eligible merchandise credits comprise 1,473 cancellation proxies, 5,883 return proxies, 1,147 unmatched credits with a known customer and 162 anonymous credits. No matched credit in this workbook has an unknown region. The historical raw worksheet `C_invoices` still contains all 9,288 C lines, including fees and duplicates. Reconcile the regional table to the eligible total; filter `credit_allocations` by Excel row for original quantities, dates, prices, lag and the remaining allocation balance.

### 将证据与地区阈值的解释分开

8,665 条符合商品口径的贷项中，操作性取消 1,473 条，操作性退货 5,883 条，已知客户未匹配 1,147 条，匿名 162 条。本工作簿中没有已匹配且地区未知的贷项。历史原表口径的 `C_invoices` 仍含全部 9,288 条 C 行，包括费用、重复记录等。可核对地区汇总与入选总数，并在 `credit_allocations` 中按 Excel 行号查看数量、时间、两种单价、时间差和可冲余额。

In [43]:
allocations = tables["credit_allocations"]
display(tables["credit_summary"])
display(allocations.groupby("status").agg(
    rows=("credit_excel_row", "size"), signed_gbp=("credit_amount_gbp", "sum"),
    units_credited=("credited_units", "sum")
))
allocation_fields = [
    "credit_excel_row", "credit_invoice_no", "purchase_excel_row", "purchase_invoice_no",
    "region", "lag_minutes", "threshold_hours", "credited_units", "balance_before", "balance_after", "status",
]
display(allocations[allocation_fields].head(8))

,region,status,rows,signed_amount_gbp
48,Europe,anonymous,49,-1190.19
49,Europe,cancellation,193,-11480.38
50,Europe,return,780,-12167.58
51,Europe,unmatched,92,-2515.96
52,ROW,anonymous,2,-216.69
53,ROW,cancellation,176,-2311.48
54,ROW,return,41,-2702.88
55,ROW,unmatched,5,-255.19
56,UK,anonymous,111,-2743.57
57,UK,cancellation,1104,-298020.28


,rows,signed_gbp,units_credited
status,,,
anonymous,162,-4150.45,2306
cancellation,1473,-311812.14,189315
return,5883,-133795.18,58386
unmatched,1147,-26113.39,20007


,credit_excel_row,credit_invoice_no,purchase_excel_row,purchase_invoice_no,region,lag_minutes,threshold_hours,credited_units,balance_before,balance_after,status
0,156,C536383,<NA>,None,UK,NaN,72,1,<NA>,<NA>,unmatched
1,237,C536391,<NA>,None,UK,NaN,72,12,<NA>,<NA>,unmatched
2,238,C536391,<NA>,None,UK,NaN,72,24,<NA>,<NA>,unmatched
3,239,C536391,<NA>,None,UK,NaN,72,24,<NA>,<NA>,unmatched
4,240,C536391,<NA>,None,UK,NaN,72,24,<NA>,<NA>,unmatched
5,241,C536391,<NA>,None,UK,NaN,72,12,<NA>,<NA>,unmatched
6,242,C536391,<NA>,None,UK,NaN,72,12,<NA>,<NA>,unmatched
7,243,C536391,<NA>,None,UK,NaN,72,24,<NA>,<NA>,unmatched


### Explain the difference from the reported 8,668 credits

The five-digit, inclusive, independent control reproduces the discussion's four class totals and signed amounts. The frozen merchandise register removes three credits on numeric carriage/packing codes. Enforcing a remaining balance leaves a further 35 identified credits unmatched compared with independent allocation under the same register. Under the canonical policy there are 12 equal-minute matches. Compare `frozen_balance_strict` with `frozen_balance_inclusive` to see which Excel rows, purchase matches or classifications change. A repeated or depleted sale cannot supply unlimited units.

Select “all” in `credit_controls` for the overall comparison; the regional entries use separate denominators. `credit_control_rows` gives individual credits and before/after purchase row, status, lag and balance. `sku_policy_differences` lists original rows whose inclusion differs, including `PADS` and the numeric service codes.

### 为什么与讨论稿中的 8,668 条贷项不同？

五位数字、允许同时间、独立匹配的对照可复现讨论稿中四类结果的行数及带符号金额。冻结商品码规则排除了三条数字编码运费或包装费贷项。在同一商品规则下，设置可冲余额后，比独立匹配多出 35 条已知客户未匹配。项目最终口径有 12 条同分钟匹配。比较 `frozen_balance_strict` 与 `frozen_balance_inclusive`，即可按 Excel 行号核对对应销售行或类别发生了哪些变化。已耗尽的购买数量无法反复用于匹配。

在 `credit_controls` 中通过 “all” 查看整体对照，分地区记录另有各自的分母。`credit_control_rows` 列出逐贷项的前后销售行、状态、时间差和余额。`sku_policy_differences` 给出商品界定不同的原始行，包括 `PADS` 及数字编码的运费和包装费。

In [44]:
controls = tables["credit_controls"]
overall = controls.loc[controls["scope"].eq("all")]
display(overall.pivot(index="metric", columns="scenario", values="observed_rows"))
display(overall.loc[overall["scenario"].eq("frozen_balance_inclusive")])
equal_minute = allocations.loc[allocations["same_time"]]
display(equal_minute[allocation_fields + ["later_excel_row_same_time"]])
changes = tables["credit_control_rows"]
display(changes.groupby(["before_scenario", "after_scenario"], sort=False).size().rename("changed_credit_rows"))
display(changes[["credit_excel_row", "before_scenario", "after_scenario", "before_status", "after_status",
                 "before_purchase_excel_row", "after_purchase_excel_row"]].head(8))
sku_changes = tables["sku_policy_differences"]
display(sku_changes.groupby(["StockCode", "Description", "five_digit_decision", "frozen_decision"],
                            dropna=False).agg(rows=("ExcelRow", "size"), signed_gbp=("signed_amount_gbp", "sum")))

scenario,five_digit_independent_inclusive,five_digit_independent_strict,frozen_balance_inclusive,frozen_balance_strict,frozen_independent_inclusive
metric,,,,,
anonymous,162.0,162.0,162.0,162.0,162.0
cancellation,1481.0,1476.0,1473.0,1466.0,1479.0
credit,8668.0,8668.0,8665.0,8665.0,8665.0
return,5913.0,5918.0,5883.0,5888.0,5912.0
unmatched,1112.0,1112.0,1147.0,1149.0,1112.0
view_a,531172.0,531172.0,531112.0,531112.0,531112.0
view_b,399656.0,399656.0,399563.0,399563.0,399563.0


,scenario,scope,metric,reported_rows,observed_rows,delta_rows,reported_amount_gbp,observed_amount_gbp,delta_amount_gbp,size_relative_to_original,return_credit_rows,distinct_linked_sale_lines,later_observed,observed_rate
28,frozen_balance_inclusive,all,view_a,531172.0,531112.0,-60.0,9770919.71,9770043.163,-876.547,NaN,<NA>,NaN,<NA>,<NA>
29,frozen_balance_inclusive,all,view_b,399656.0,399563.0,-93.0,8265476.93,8264201.933,-1274.997,NaN,<NA>,NaN,<NA>,<NA>
30,frozen_balance_inclusive,all,credit,8668.0,8665.0,-3.0,NaN,-475871.160,NaN,NaN,<NA>,NaN,<NA>,<NA>
31,frozen_balance_inclusive,all,cancellation,1481.0,1473.0,-8.0,-312072.49,-311812.140,260.350,NaN,<NA>,NaN,<NA>,<NA>
32,frozen_balance_inclusive,all,return,5913.0,5883.0,-30.0,-134674.78,-133795.180,879.600,NaN,<NA>,NaN,<NA>,<NA>
33,frozen_balance_inclusive,all,unmatched,1112.0,1147.0,35.0,-25003.44,-26113.390,-1109.950,NaN,<NA>,NaN,<NA>,<NA>
34,frozen_balance_inclusive,all,anonymous,162.0,162.0,0.0,-4150.45,-4150.450,0.000,NaN,<NA>,NaN,<NA>,<NA>


,credit_excel_row,credit_invoice_no,purchase_excel_row,purchase_invoice_no,region,lag_minutes,threshold_hours,credited_units,balance_before,balance_after,status,later_excel_row_same_time
139,13021,C537408,13015,537407,UK,0.0,72,6,6,0,cancellation,False
140,13022,C537408,13020,537407,UK,0.0,72,12,12,0,cancellation,False
1001,62714,C541512,62713,541513,UK,0.0,72,6,12,6,cancellation,False
1693,94833,C544429,94832,544430,UK,0.0,72,2,2,0,cancellation,False
2138,125233,C547031,125206,547029,UK,0.0,72,8,32,24,cancellation,False
2668,155039,C549962,155038,549961,UK,0.0,72,2,2,0,cancellation,False
2836,164797,C550672,164796,550671,Europe,0.0,120,100,100,0,cancellation,False
6099,365082,C568694,365080,568695,ROW,0.0,240,12,12,0,cancellation,False
6100,365083,C568694,365081,568695,ROW,0.0,240,24,24,0,cancellation,False
7800,466264,C576324,466235,576323,UK,0.0,72,1,2,1,cancellation,False


before_scenario                   after_scenario                  
five_digit_independent_strict     five_digit_independent_inclusive     12
five_digit_independent_inclusive  frozen_independent_inclusive          3
frozen_independent_inclusive      frozen_balance_inclusive            287
frozen_balance_strict             frozen_balance_inclusive             12
Name: changed_credit_rows, dtype: int64

,credit_excel_row,before_scenario,after_scenario,before_status,after_status,before_purchase_excel_row,after_purchase_excel_row
0,13021,five_digit_independent_strict,five_digit_independent_inclusive,cancellation,cancellation,9771,13015
1,13022,five_digit_independent_strict,five_digit_independent_inclusive,cancellation,cancellation,9773,13020
2,62714,five_digit_independent_strict,five_digit_independent_inclusive,cancellation,cancellation,62702,62713
3,94833,five_digit_independent_strict,five_digit_independent_inclusive,return,cancellation,87290,94832
4,125233,five_digit_independent_strict,five_digit_independent_inclusive,return,cancellation,110139,125206
5,155039,five_digit_independent_strict,five_digit_independent_inclusive,return,cancellation,117394,155038
6,164797,five_digit_independent_strict,five_digit_independent_inclusive,cancellation,cancellation,158298,164796
7,365082,five_digit_independent_strict,five_digit_independent_inclusive,cancellation,cancellation,364986,365080


rows  \
StockCode Description                  five_digit_decision frozen_decision             
23444     Next Day Carriage            credit              excluded_stock_code     1   
                                       sale                excluded_stock_code    79   
23574     PACKING CHARGE               credit              excluded_stock_code     2   
                                       sale                excluded_stock_code    14   
DCGS0003  BOXED GLASS ASHTRAY          excluded_stock_code sale                    4   
DCGS0004  HAYNES CAMPER SHOULDER BAG   excluded_stock_code sale                    1   
DCGS0069  OOH LA LA DOGS COLLAR        excluded_stock_code sale                    1   
DCGS0070  CAMOUFLAGE DOG COLLAR        excluded_stock_code sale                    1   
DCGS0076  SUNJAR LED NIGHT NIGHT LIGHT excluded_stock_code sale                    2   
DCGSSBOY  BOYS PARTY BAG               excluded_stock_code sale                   11   
DCGSSGIRL GIRLS PARTY BAG              excluded_stock_code sale                   13   
PADS      PADS TO MATCH ALL CUSHIONS   excluded_stock_code sale                    3   

                                                                                signed_gbp  
StockCode Description                  five_digit_decision frozen_decision                  
23444     Next Day Carriage            credit              excluded_stock_code     -15.000  
                                       sale                excluded_stock_code    1200.000  
23574     PACKING CHARGE               credit              excluded_stock_code     -15.000  
                                       sale                excluded_stock_code     105.000  
DCGS0003  BOXED GLASS ASHTRAY          excluded_stock_code sale                      9.940  
DCGS0004  HAYNES CAMPER SHOULDER BAG   excluded_stock_code sale                     16.630  
DCGS0069  OOH LA LA DOGS COLLAR        excluded_stock_code sale                     15.790  
DCGS0070  CAMOUFLAGE DOG COLLAR        excluded_stock_code sale                     12.720  
DCGS0076  SUNJAR LED NIGHT NIGHT LIGHT excluded_stock_code sale                     48.390  
DCGSSBOY  BOYS PARTY BAG               excluded_stock_code sale                    150.550  
DCGSSGIRL GIRLS PARTY BAG              excluded_stock_code sale                    144.430  
PADS      PADS TO MATCH ALL CUSHIONS   excluded_stock_code sale                      0.003

### Which units disappear, and when?

* Full return: sale 569206 row 370990 had 6 units. C573097 consumed all 6. Partial return: sale 569207 row 371018 had 24. C572440 left 14. Mixed sale 539395 row 34475 began at 36; C539644 and C540158 deducted 2 and 5, leaving 29.
* On September invoice 565141, row 321282, one unit of SKU 21686 was fully reversed. Another invoice, 567473 row 348818, retained one unit. The same customer can therefore still have a positive September label for this SKU. October invoice 569329 row 373424 similarly lost all 3 units, followed by a separate sale on invoice 573104 row 422680.
* Time matters: row 5249 had 32 units at the October 1 training cutoff. C580954 on December 6 removed them in the December view. The two large pairs 541431/C541433 and 581483/C581484 reached zero (74,215 and 80,995 units). The December pair was not yet observed at October 1.
* On C556448, credit row 222694 records −60 × £4.95 = −£297. Matched sale row 222682 records 60 × £649.50 = £38,970. Across 28 unequal-price cancellation matches and the other cancellation matches, booked credits total −£311,812.14 and the corresponding original-price deduction is £349,595.90. This cancellation-only accounting subtotal covers one operational class. Net purchase value deducts units from every matched credit at the original sale price.

### 哪些数量会消失，在什么时候消失？

* 全退：销售单 569206 第 370990 行原有 6 件，C573097 将其全部冲销。部分退：569207 第 371018 行原有 24 件，C572440 后剩 14 件。混合贷项：539395 第 34475 行原有 36 件，C539644 和 C540158 先后扣去 2 件及 5 件，剩 29 件。
* 九月发票 565141 第 321282 行的商品 21686 共 1 件，已全部冲销。另一张发票 567473 第 348818 行仍保留 1 件，所以该客户九月该 SKU 的正例仍然成立。十月发票 569329 第 373424 行的 3 件也全部冲销，后来另有销售单 573104 第 422680 行重新购买。
* 十月一日训练截止时，第 5249 行仍有 32 件。十二月六日的 C580954 才让这条销售行在十二月视图归零。另两组大额撤销 541431/C541433 和 581483/C581484 分别从 74,215 件和 80,995 件归零。十二月的第二组在十月一日尚不可见。
* C556448 第 222694 行贷项记录 −60 × £4.95 = −£297，匹配销售第 222682 行记录 60 × £649.50 = £38,970。全部取消匹配中有 28 条的贷项与原销售单价不同。全体取消贷项账面合计 −£311,812.14，原售价扣减为 £349,595.90。这笔仅扣取消的金额是该操作类别的账务小计。购买净额按原售价扣除每一笔已匹配贷项的数量。

In [45]:
samples = tables["reversal_examples"]
print(samples[allocation_fields + ["credit_unit_price", "purchase_unit_price", "gross_units_after_cancellations",
                                     "purchase_units_after_credit", "purchase_units_at_oct1"]].to_string(index=False))
display(raw.loc[raw["ExcelRow"].isin([321282, 348818, 373424, 422680]), ["ExcelRow", "CustomerID", "StockCode", "InvoiceNo", "InvoiceDate", "Quantity", "UnitPrice"]])
linked = allocations.dropna(subset=["purchase_excel_row"]).copy()
cancellation = linked.loc[linked["status"].eq("cancellation")]
print("Unequal cancellation prices / 取消贷项与销售单价不同", cancellation["credit_unit_price"].ne(cancellation["purchase_unit_price"]).sum())
print("Signed cancellation GBP / 取消贷项金额", round(cancellation["credit_amount_gbp"].sum(), 2))
print("Original-price cancellation deduction GBP / 原销售价取消扣减额",
      round((cancellation["credited_units"] * cancellation["purchase_unit_price"]).sum(), 2))
linked["sale_month"] = linked["purchase_time"].dt.to_period("M")
linked["credit_month"] = linked["credit_time"].dt.to_period("M")
cross_month = linked.loc[linked["sale_month"].ne(linked["credit_month"])]
display(cross_month.groupby(["status", "sale_month", "credit_month"]).size().rename("credit_rows").reset_index().tail(12))

 credit_excel_row credit_invoice_no  purchase_excel_row purchase_invoice_no region  lag_minutes  threshold_hours  credited_units  balance_before  balance_after       status  credit_unit_price  purchase_unit_price  gross_units_after_cancellations  purchase_units_after_credit  purchase_units_at_oct1
              941           C536506                 886              536488     UK          7.0               72               6               8              2 cancellation               4.25                 4.25                                2                            2                       2
            16426           C537656                  67              536374     UK       9072.0               72              12              32             20       return              10.95                10.95                               32                           20                      20
            39447           C539644               34475              539395 Europe       4560.0        

,ExcelRow,CustomerID,StockCode,InvoiceNo,InvoiceDate,Quantity,UnitPrice
321280,321282,17841,21686,565141,2011-09-01 13:40:00,1,39.95
348816,348818,17841,21686,567473,2011-09-20 12:50:00,1,39.95
373422,373424,17652,23494,569329,2011-10-03 13:03:00,3,5.95
422678,422680,17652,23494,573104,2011-10-27 14:59:00,3,5.95


Unequal cancellation prices / 取消贷项与销售单价不同 28
Signed cancellation GBP / 取消贷项金额 -311812.14
Original-price cancellation deduction GBP / 原销售价取消扣减额 349595.9


,status,sale_month,credit_month,credit_rows
70,return,2011-07,2011-10,35
71,return,2011-07,2011-11,19
72,return,2011-08,2011-09,127
73,return,2011-08,2011-10,34
74,return,2011-08,2011-11,28
75,return,2011-08,2011-12,10
76,return,2011-09,2011-10,351
77,return,2011-09,2011-11,63
78,return,2011-09,2011-12,4
79,return,2011-10,2011-11,317


### How many original lines and basket items does this affect?

The starting population is 391,060 identified positive merchandise sale lines across the workbook. After cancellations alone, 390,168 remain. Deducting all matched credits leaves 388,476 lines and 383,669 basket–SKU events. Of the 1,692 fully depleted original sale lines involving returns (0.433% of 391,060), 1,689 received returns alone and 3 received mixed credits. Only 1,683 additional basket–SKU events disappear relative to cancellation-only gross. A further 3,999 return-affected sale lines keep positive units. The training rows in the table replay credits available strictly before October 1.

### 影响了多少原销售行及篮内商品？

全期起点为已知客户的 391,060 条正向商品销售明细。仅扣取消后剩 390,168 行；扣除全部匹配贷项后剩 388,476 行、383,669 个篮内商品事件。涉及退货并累计归零的原销售有 1,692 行，占 391,060 行的 0.433%，其中 1,689 行仅有退货，3 行同时有取消和退货。相比仅扣取消，最终消失的篮内商品事件为 1,683 个，因为同一篮内同一商品可能还有其他销售行。另有 3,999 条涉及退货的销售仍保留正数量。训练口径只重放十月一日以前已出现的贷项。

In [46]:
display(tables["retention_comparison"])
transitions = tables["retention_transitions"]
display(transitions.loc[transitions["window"].eq("whole_workbook")])

,definition,sale_lines,basket_sku_events,baskets,customers,units,value_at_sale_prices_gbp,window
0,observed_positive_sales,262597,260199,13081,3613,3604195,6085531.503,training_before_october
1,gross_after_cancellations,262067,259676,13040,3612,3514634,5935091.803,training_before_october
2,net_after_all_matched_credits,260919,258532,13001,3608,3472284,5837037.733,training_before_october
3,observed_positive_sales,391060,386236,18402,4334,5141694,8735922.643,whole_workbook
4,gross_after_cancellations,390168,385352,18331,4330,4952379,8386326.743,whole_workbook
5,net_after_all_matched_credits,388476,383669,18280,4323,4893993,8252857.003,whole_workbook


,unit,state,credit_composition,count,observed_units,retained_units,all_observed,share_of_all_observed,window
14,source_sale_line,full,cancellation,892,184963,0,391060,0.002281,whole_workbook
15,source_sale_line,full,mixed,3,12,0,391060,0.000008,whole_workbook
16,source_sale_line,full,return,1689,37143,0,391060,0.004319,whole_workbook
17,source_sale_line,partial,cancellation,541,12687,8487,391060,0.001383,whole_workbook
18,source_sale_line,partial,mixed,24,933,624,391060,0.000061,whole_workbook
19,source_sale_line,partial,return,3975,88970,67896,391060,0.010165,whole_workbook
20,source_sale_line,unaffected,none,383936,4816986,4816986,391060,0.981783,whole_workbook
21,basket_sku_event,full,cancellation,884,184775,0,386236,0.002289,whole_workbook
22,basket_sku_event,full,mixed,3,12,0,386236,0.000008,whole_workbook
23,basket_sku_event,full,return,1680,37118,0,386236,0.004350,whole_workbook


### Where do 1,686 and 37.4% come from?

In the historical five-digit, inclusive, independently reusable matching control, 1,686 of 5,913 return **credit rows** individually equal their original sale quantity. These rows reference 1,669 distinct sales. The 4,227 other return credits are individually partial. After 1,582 of those, another invoice records a strictly later paid purchase of the same SKU by the same customer (37.43%). Under the frozen SKU rules with a remaining balance, the corresponding results are 1,663 individually full and 1,579 of 4,220 individually partial (37.42%). A round percentage can agree while both the numerator and denominator differ.

At the **original sale** level, 3,999 partially retained lines received a return. Following up after their first return, 1,495 have any later paid purchase (37.38%). Following up after the last return and requiring the later purchase to retain units at workbook end yields 1,483 (37.08%). Across all 4,540 partially credited lines, including cancellation-only lines, the last matched credit is followed by an observed sale for 1,717 or a net-positive sale for 1,708. Some sales had multiple credits; per-credit percentages can weight them more than once. These are observed same-SKU purchases up to December 9. Later credits have shorter follow-up. The figures cannot establish what caused repurchase or guarantee retention of a later positive label.

### 1,686 与 37.4% 的统计口径是什么？

在历史“五位数字、允许同时间、各贷项可独立复用同一销售”对照中，5,913 条操作性退货**贷项行**中的 1,686 条，单笔数量等于原销售数量。这些贷项对应 1,669 条不同的销售。其余 4,227 条单笔数量小于原销售数量；其中 1,582 条之后，同一客户在另一张发票上严格更晚再次购买了同一商品（37.43%）。换成冻结商品规则和剩余数量约束，单笔全退为 1,663 条，单笔部分退之后再购买为 1,579 / 4,220（37.42%）。四舍五入后的比例可以相同，但分子与分母都有变化。

改按**原销售行**计，3,999 条涉及退货的明细在期末仍有部分数量。从该行首次退货起，之后有任意付费再购买的是 1,495 条（37.38%）。从末次退货起，之后再购买且后买行到期末仍有正余额的是 1,483 条（37.08%）。若把仅取消也纳入，在全部 4,540 条部分冲销行中，从末次匹配贷项起，1,717 条之后有销售，1,708 条之后有期末净正的销售。逐贷项统计会让接收多笔贷项的销售被计入多次。观察止于十二月九日，越接近期末的退货，追踪时间越短。上述数据用于描述再次购买，无法推断其成因或保证将来的正例。

In [47]:
credit_rows = controls.loc[controls["scope"].eq("individual_return_credit_rows")]
display(credit_rows[["scenario", "size_relative_to_original", "return_credit_rows",
                     "distinct_linked_sale_lines", "later_observed", "observed_rate"]])
display(tables["return_repurchase"])

,scenario,size_relative_to_original,return_credit_rows,distinct_linked_sale_lines,later_observed,observed_rate
80,five_digit_independent_inclusive,full,1686,1669.0,401,0.237841
81,five_digit_independent_inclusive,partial,4227,4031.0,1582,0.374261
82,frozen_balance_inclusive,full,1663,1663.0,389,0.233915
83,frozen_balance_inclusive,partial,4220,4028.0,1579,0.374171


,anchor,source_outcome,denominator,distinct_source_sales,later_observed,later_net,unit,observed_rate,net_rate
0,each_operational_return,full,1720,1692,403,401,return_credit_row,0.234302,0.23314
1,each_operational_return,partial,4163,3999,1565,1557,return_credit_row,0.375931,0.374009
2,first_operational_return,full,1692,1692,396,394,source_sale_line,0.234043,0.232861
3,first_operational_return,partial,3999,3999,1495,1487,source_sale_line,0.373843,0.371843
4,last_matched_credit,full,2584,2584,772,767,source_sale_line,0.298762,0.296827
5,last_matched_credit,partial,4540,4540,1717,1708,source_sale_line,0.378194,0.376211
6,last_operational_return,full,1692,1692,396,394,source_sale_line,0.234043,0.232861
7,last_operational_return,partial,3999,3999,1491,1483,source_sale_line,0.372843,0.370843


## Build the worksheet guide and check the export order

The guide lists the other 26 sheets. Check their names against the tables constructed above. The exported detail tabs retain every qualifying row, including those beyond the notebook previews.

## 生成工作表目录并核对导出顺序

目录介绍其余 26 张表。把目录中的名称与上方代码生成的表逐一核对后再导出。明细表保留所有命中行，包含 notebook 预览以外的行。

In [48]:
guide = [
    ("sign_combinations", "Disjoint cells by C prefix, quantity sign, price sign and customer ID presence. Counts cover every raw line.", "按 C 前缀、数量及价格符号、客户编号有无交叉分组。各组合互斥，合计覆盖原表。"),
    ("missingness", "Missing fields, affected invoice counts, and intersection with missing customer IDs on the original sheet.", "原表各字段缺失行数、涉及的不同发票数及与缺客户编号的交集。"),
    ("customer_id_invoices", "Full invoice lookup for each invoice with a missing customer ID, including counts of known IDs elsewhere on the invoice.", "逐张回查含缺失客户编号的发票，统计同票其他行出现的不同已知客户编号。"),
    ("filter_attribution", "Attribute each row in a scope to the first applicable purchase filter or to retained. Scopes overlap. This is a count simulation.", "对每个筛选范围的行，统计最先命中的购买过滤规则或保留数。范围可重叠，仅作计数模拟。"),
    ("C_invoices", "All lines on C-prefixed invoices, including discounts and other non-merchandise lines.", "全部 C 前缀明细，含折扣和其他非商品记录。"),
    ("Q_nonpositive", "All lines with Quantity ≤ 0, including C invoices.", "数量 ≤ 0 的全部明细，包括 C 前缀记录。"),
    ("P_nonpositive", "All lines with UnitPrice ≤ 0. Filter price_sign to separate zero and negative recorded prices.", "单价 ≤ 0 的全部明细。用 price_sign 区分零价与负价。"),
    ("nonC_negative", "All non-C negative quantities. ExcelRow counts the header as row 1. Original descriptions and IDs are preserved.", "全部非 C 负数量明细。ExcelRow 对应原表行号，表头为第 1 行。保留原描述与编号。"),
    ("history_summary", "Counts of flagged rows with at least one match. Earlier means strictly before the flagged timestamp. Positive means Quantity > 0. Unknown customer keys are blank in row-level evidence.", "汇总至少有一条匹配的明细数。历史匹配严格早于当前时间。正数量为 Quantity > 0。明细中客户匹配留空表示无法比较。"),
    ("note_summary", "Counts by exact combination of note/code clues in each scope. Unknown means no recognized wording. Scopes overlap.", "按备注与特殊编码线索的组合计数。unknown 表示未识别到关键词。不同筛选范围之间有重叠。"),
    ("case_examples", "Selected original entries in chronological context. Signed amount is quantity times unit price in GBP.", "从原表选取的交易及上下文。带符号金额为数量乘单价，单位英镑。"),
    ("invoice_references", "Invoice-shaped tokens in the Description and exact matches in the workbook, with earlier-line counts.", "备注里的发票号形式文本，在原表中精确回查，并统计较早的明细。"),
    ("sku_names", "Exact stored labels and codes for a name lookup after trimming surrounding whitespace. Product identity remains unverified.", "按名称去除首尾空白查找，展示原始名称和编码。商品是否完全相同仍待商品主数据核实。"),
    ("amount_by_record_type", "Signed line arithmetic by exception status and customer ID availability. Actual payments and refunds are unobserved.", "按标记与有无客户编号汇总带符号行金额。原表没有付款到账或退款流水。"),
    ("customer_impact_summary", "Identified customers grouped by whether they have regular, exceptional or both kinds of lines.", "按已知客户拥有常规行、异常行或两类兼有进行汇总。"),
    ("customer_impact", "Per identified customer, compare distinct invoices, recency and monetary totals under both definitions. Full-window descriptive analysis.", "对每一位已知客户，比较两种口径的不同发票数、距最近记录天数及金额。仅作全期描述。"),
]
REVERSAL_GUIDE = [
    ("ledger_summary", "Frozen merchandise policy. Signed sales and credits retain recorded prices. At original sale prices, gross deducts cancellations only and net deducts every matched credit. Source exclusions are first-match counts.", "冻结商品规则。带符号销售与贷项取各行记录价格；按原销售单价，毛额只扣匹配取消，净额扣除所有可匹配贷项。来源筛选按最先命中的规则归因。"),
    ("credit_summary", "Canonical merchandise credits by destination region and operational status. Amounts use recorded credit prices. Time thresholds are regional assumptions.", "按贷项目的地区与操作分类汇总实际规则下的商品贷项，金额取贷项单价。时限是项目对地区的假设。"),
    ("credit_allocations", "Every eligible merchandise credit with matched sale row, time lag, region, threshold and allocation balance. An unmatched row has no allocated sale.", "全部符合条件的商品贷项，列出匹配销售行、时间差、地区、时限及分配余额；未匹配的贷项无对应销售行。"),
    ("credit_controls", "Compare SKU, timing and allocation assumptions with the discussion benchmarks, including per-credit full/partial counts and later same-SKU purchases.", "对照商品码、时间和贷项分配方式，复核讨论稿给出的逐条全退、部分退和后来同客同货的购买比例。"),
    ("credit_control_rows", "Credit rows whose inclusion, matched sale or class changes between scenarios. Use the source row numbers to reproduce a difference.", "对照情景之间在纳入状态、对应销售行或操作类别上有变化的贷项，可用原始行号复核。"),
    ("reversal_examples", "Selected credit and sale links including two large reversals, unequal prices, partial cancellations and returns. Remaining allocation capacity equals net retained units after each matched credit.", "追踪两笔大额撤销、价格不相等、部分取消及退货示例的贷项与销售。每次匹配后，可冲余额即净保留数量。"),
    ("sku_policy_differences", "Source rows with different decisions under the five-leading-digit control and frozen product-code rules, including PADS and numeric delivery or packing codes.", "五位数字开头对照规则与冻结商品清单产生不同决定的原始行，含 PADS 及数字编码的运费、包装费。"),
    ("retention_comparison", "Identified positive lines, basket–SKU events, units and original-price values under observed, cancellation-only gross and net definitions. Each window has its own credit cutoff.", "按原销售、仅扣取消的毛额和扣所有匹配贷项的净额，对比已知客户的销售行、篮内商品事件、数量和原价金额；每个窗口单独设贷项截止点。"),
    ("retention_transitions", "Full, partial and unaffected source sales and basket–SKU events, split by credit composition. Each share uses all observed units of that grain in the window.", "按来源销售行和篮内商品两种粒度，分别统计全部冲销、部分冲销和未受影响的数量及贷项构成。各占比以对应窗口该粒度全部记录为分母。"),
    ("return_repurchase", "Full-workbook follow-up after first, last or individual return credits or the last matched credit. Compare a later observed sale with a later sale still net-positive at the end.", "全期资料中，分别从首次、末次、每次操作性退货或末次匹配贷项追查，同客同货其他订单其后是否有销售，以及是否有到期末仍为净正的销售。"),
]
guide.extend(REVERSAL_GUIDE)
tables["guide"] = pd.DataFrame(guide, columns=["Sheet", "What it contains", "内容与口径"])
sheet_order = [
    "guide", "sign_combinations", "missingness", "customer_id_invoices", "filter_attribution",
    "history_summary", "note_summary", "case_examples", "C_invoices", "Q_nonpositive",
    "P_nonpositive", "nonC_negative", "invoice_references", "sku_names",
    "amount_by_record_type", "customer_impact_summary", "customer_impact",
]
sheet_order.extend(name for name, _, _ in REVERSAL_GUIDE)
assert set(tables["guide"]["Sheet"]) == set(sheet_order) - {"guide"}
assert set(tables) == set(sheet_order) and len(tables) == 27
tables = {name: tables[name] for name in sheet_order}
display(tables["guide"])

,Sheet,What it contains,内容与口径
0,sign_combinations,"Disjoint cells by C prefix, quantity sign, pri...",按 C 前缀、数量及价格符号、客户编号有无交叉分组。各组合互斥，合计覆盖原表。
1,missingness,"Missing fields, affected invoice counts, and i...",原表各字段缺失行数、涉及的不同发票数及与缺客户编号的交集。
2,customer_id_invoices,Full invoice lookup for each invoice with a mi...,逐张回查含缺失客户编号的发票，统计同票其他行出现的不同已知客户编号。
3,filter_attribution,Attribute each row in a scope to the first app...,对每个筛选范围的行，统计最先命中的购买过滤规则或保留数。范围可重叠，仅作计数模拟。
4,C_invoices,"All lines on C-prefixed invoices, including di...",全部 C 前缀明细，含折扣和其他非商品记录。
5,Q_nonpositive,"All lines with Quantity ≤ 0, including C invoi...",数量 ≤ 0 的全部明细，包括 C 前缀记录。
6,P_nonpositive,All lines with UnitPrice ≤ 0. Filter price_sig...,单价 ≤ 0 的全部明细。用 price_sign 区分零价与负价。
7,nonC_negative,All non-C negative quantities. ExcelRow counts...,全部非 C 负数量明细。ExcelRow 对应原表行号，表头为第 1 行。保留原描述与编号。
8,history_summary,Counts of flagged rows with at least one match...,汇总至少有一条匹配的明细数。历史匹配严格早于当前时间。正数量为 Quantity > 0。明...
9,note_summary,Counts by exact combination of note/code clues...,按备注与特殊编码线索的组合计数。unknown 表示未识别到关键词。不同筛选范围之间有重叠。


## Export and sources

Write all tables to `results/transaction_exceptions.xlsx`. In Excel, filter by `ExcelRow`, `InvoiceNo`, or `StockCode` to trace a row. The original eight fields are present in the full case sheets. Chen, Sain and Guo (2012), pp. 199–203, discuss transaction preparation and RFM customer segmentation. Their analysis supplies context for the customer-activity comparison. The counts and case evidence here come directly from the identified workbook. [Chen, Sain & Guo (2012)](https://doi.org/10.1057/dbm.2012.17) · [Chen (2015), UCI Online Retail, CC BY 4.0](https://doi.org/10.24432/C5BW33)

## 导出与来源

将全部表格写入 `results/transaction_exceptions.xlsx`。在 Excel 中可用 `ExcelRow`、`InvoiceNo` 或 `StockCode` 筛选和追溯明细，完整案例工作表保留八个原始字段。Chen、Sain 和 Guo（2012，第 199–203 页）讨论了交易数据准备与基于 RFM 的客户分群，为此次客户活动口径比较提供背景。本次统计量和案例证据均取自已校验的原始工作簿。[Chen、Sain 和 Guo（2012）](https://doi.org/10.1057/dbm.2012.17) · [Chen（2015），UCI Online Retail，CC BY 4.0](https://doi.org/10.24432/C5BW33)

In [49]:
from tempfile import TemporaryDirectory
from openpyxl.utils import get_column_letter

def write_workbook(tables: dict[str, pd.DataFrame], output: Path) -> None:
    """Write filterable sheets, preserving original descriptions as literal text."""
    path = Path(output)
    if path.suffix.lower() != ".xlsx":
        raise ValueError("Choose an .xlsx output file.")
    path.parent.mkdir(parents=True, exist_ok=True)
    with TemporaryDirectory(dir=path.parent) as work:
        temp = Path(work) / path.name
        with pd.ExcelWriter(temp, engine="openpyxl", datetime_format="yyyy-mm-dd hh:mm:ss") as writer:
            for name, frame in tables.items():
                frame.to_excel(writer, sheet_name=name, index=False)
                sheet = writer.sheets[name]
                sheet.freeze_panes = "B2"
                sheet.auto_filter.ref = sheet.dimensions
                for number, col in enumerate(frame.columns, start=1):
                    width = 48 if col == "Description" else max(17, len(col) + 3)
                    if name == "guide":
                        width = 32 if number == 1 else 105
                    sheet.column_dimensions[get_column_letter(number)].width = width
                # A leading '=' in an original description is data, not an Excel formula.
                for row in sheet.iter_rows(min_row=2):
                    for cell in row:
                        if cell.data_type == "f":
                            cell.data_type = "s"
        temp.replace(path)

write_workbook(tables, output)
print(f"Saved {len(tables)} sheets / 已保存 {len(tables)} 张工作表：{output if drive else output.relative_to(project)}")

Saved 27 sheets / 已保存 27 张工作表：results/transaction_exceptions.xlsx
